# 🧠 Customer Return Prediction & AI Calling Prioritization

A portfolio-safe, end-to-end machine learning pipeline for predicting 30-day customer return probability, building marketing recommendation layers, and prioritizing customers for AI voice sales outreach.

### Pipeline
**Raw Orders → Cleaning → Order-Level Features → 30-Day Return Target → Random Forest → Customer Scoring → Marketing Targeting → Lead Prioritization → Voice AI Call Brief**

> **Privacy note:** The production dataset is intentionally not included. Outputs containing customer IDs, phone numbers, order IDs, or other customer-level records were removed from this public notebook.


In [ ]:
# =========================
# Block 1: Read Data File
# =========================
import pandas as pd
import os


In [ ]:
DATA_PATH = os.getenv("CUSTOMER_DATA_PATH", "data/customer_orders_sample.xlsx")
file = DATA_PATH
df = pd.read_excel(file)
print("File shape" , df.shape)
print("\nFile Columns")
print(df.columns.tolist())


## 1. Data Cleaning & Type Preparation

Clean source columns, normalize data types, and remove records missing critical order/customer fields.


In [ ]:
# =========================
# Block 2: Clean columns
# =========================
df = df.loc[: , ~df.columns.astype(str).str.contains("^Unname").copy()]
df.columns=(df.columns.astype(str).str.strip())
print("Data Shape After Cleaning : " , df.shape)
print(df.columns.tolist())
display(df.head())


In [ ]:
# =========================
# Block 3: Fix data types and basic cleaning
# =========================

# تحويل تاريخ الطلب إلى تاريخ حقيقي
df['order_date'] = pd.to_datetime(
    df['order_date'],
    errors='coerce'
)

# الأعمدة الرقمية
numeric_columns = [
    'qty_ordered',
    'unit_price',
    'total',
    'express'
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    )


df['customer_id']=pd.to_numeric(df['customer_id'] , errors= 'coerce').astype('Int64').astype(str)



text_columns = [
    'order_id',
    'product_variant',
    'bag',
    'residency_type',
    'order_category',
    'service',
    'laundry',
    'laundry_type'
]

for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )

if 'customer_phone' in df.columns:
  pd.to_numeric(df["customer_phone"] , errors='coerce').astype('Int64').astype(str)

  df['cusromer_phone']= df['customer_phone'].replace('<NA>' , pd.NA)


if 'is_active' in df.columns:
  df['is_active']=df['is_active'].astype(str).str.strip()





print("Data types after conversion:")
print(df.dtypes)
important_cols = [
    'order_date',
    'order_id',
    'customer_id',
    'total',
    'customer_phone',
    'is_active'
]

important_cols=[col for col in important_cols if col in df.columns]

print("\nMissing values after basic cleaning:")
print(df[important_cols].isna().sum())


print("\nDate range:")
print(df['order_date'].min(), "to", df['order_date'].max())
print("\nRows:", len(df))

display(df.head())


In [ ]:
# =========================
# Block 4: Remove rows with missing critical values
# =========================

critical_columns = [ 'order_date','order_id','customer_id','total']

data_clean = df.dropna(subset=critical_columns).copy()

print('Rows Before Cleaning' , len(df))
print('Rows After Cleaning' , len(data_clean))
print("Row Remove" , len(df) - len(data_clean))
print("Missing values after cleaning:")
print(data_clean[critical_columns].isna().sum())
print("\nDate range after cleaning:")
print(data_clean['order_date'].min(), "to", data_clean['order_date'].max())
print("\nRows:", len(data_clean))
display(data_clean.head())


## 2. Order-Level Feature Engineering

Transform item-level transactions into order-level features, including product, service, and financial behavior.


In [ ]:
# =========================
# Block 5: Convert item-level data to order-level data
# =========================
import re

# =========================
# Helper Functions
# =========================

def normalize_missing_text_columns(df , columns):
  missing_tokens = ['nan' , 'none', 'nat' , 'null' , '']

  for col in columns:
    if col in df.columns:
      df[col]=df[col].replace(missing_tokens , pd.NA)
      df[col]=df[col].replace([tokens.upper() for tokens in missing_tokens] , pd.NA)
      df[col]=df[col].replace([tokens.capitalize for tokens in missing_tokens], pd.NA)
  return df

def first_vaild_value(series):
  value = series.dropna()
  value = value[value.astype(str).str.strip() != ""]

  if len(value)==0 :
    return pd.NA

  return value.iloc[0]


def clean_unique_join(series):
  value = series.dropna().astype(str).str.strip()
  value = value[value !=""]
  unique_values = sorted(value.unique())

  return " + ".join(unique_values) if len(unique_values) > 0 else "Unknown"


def safe_column_name(value , prefix):
  value = str(value).strip().lower()
  value = re.sub(r"[^a-zA-Z0-9]+" , "_", value)
  value = value.strip("_")

  return f"{prefix}_{value}"


def detect_fainancial_category(product_variant , total_value):
  if pd.isna(product_variant):
    product_text = ""
  else:
    product_text=str(product_variant).strip().lower()

  if "delivery fee" in product_text:
      return "delivery_fee"

  if "promocode" in product_text:
      return "promocode_discount"

  if "gift wallet" in product_text:
      return "gift_wallet_discount"

  if "wallet" in product_text:
      return "wallet_discount"

  if "loyalty" in product_text:
      return "loyalty_discount"

  if "subscription" in product_text:
      return "subscription_discount"

  if "refund" in product_text:
      return "refund"

  if "minimum order" in product_text:
      return "minimum_order"

  if "package" in product_text:
      return "package"

  if product_text in ["bag", "bag fee"]:
      return "bag_fee"

  if "discount" in product_text:
      return "other_discount"

  try:
    if pd.notna(total_value) and float(total_value) < 0:
      return "other_negative_adjustment"
  except:
    pass

  return None



def join_real_products(series):
  mask = data_clean.loc[series.index , "is_real_product_row"]
  value = series[mask].dropna().astype(str).str.strip()
  value = value[value!= ""]
  unique_values = sorted(value.unique())

  return " + ".join(unique_values) if len(unique_values) > 0 else "No Financial Items"


def join_financial_items(series):
  value = series.dropna().astype(str).str.strip()
  value = value[value!=""]
  unique_values=sorted(value.unique())

  return " + ".join(unique_values) if len(unique_values) > 0 else "No Financial Items"

def normalize_is_active(value):
  text = str(value).strip().lower()
  if text in ["true" , "1" , "yes" , "y"]:
    return True
  return False






In [ ]:
# -------------------------------------------------
# 0) تجهيز أعمدة مساعدة داخل data_clean
# -------------------------------------------------
text_like_columns = [
    "product_variant",
    "bag",
    "residency_type",
    "order_category",
    "service",
    "laundry",
    "laundry_type",
    "customer_phone",
    "is_active"
]

data_clean=normalize_missing_text_columns(data_clean , text_like_columns)

if 'is_active' in data_clean.columns:
  data_clean["has_active_subscription"] = data_clean["is_active"].apply(normalize_is_active)
else:
  data_clean["has_active_subscription"]=False

data_clean["financial_category"] = data_clean.apply(lambda row : detect_fainancial_category(row.get("product_variant") , row.get("total")) ,axis = 1 )



data_clean["is_real_product_row"]=(
    (data_clean["total"] > 0 ) &
    (data_clean["product_variant"].notna())&
    (data_clean["product_variant"].astype(str).str.strip() != "")&
    (data_clean["financial_category"].isna())

)


data_clean["financial_amount_for_feature"] = data_clean["total"].abs()




In [ ]:
# -------------------------------------------------
# 1) Order-level الأساسي
# -------------------------------------------------

order_level=data_clean.groupby("order_id").agg(
    customer_id=("customer_id" ,"first" ),
    order_date = ("order_date" , "min"),
    laundry = ("laundry" , "first"),
    laundry_type = ("laundry_type" , "first"),
    order_category = ("order_category" , "first"),
    residency_type = ("residency_type" , "first"),
    express = ("express" , "max"),
    has_active_subscription = ("has_active_subscription" , "max"),
    service = ("service" , clean_unique_join),
    customer_phone = ("customer_phone" , first_vaild_value),
    product_variants = ("product_variant" , join_real_products),
    financial_items = ("financial_category" , join_financial_items),

    bag = ("bag", lambda x : "Has Bag" if x.dropna().astype(str).str.strip().ne("").any() else "No Bag" ),
    order_revenue = ("total" , "sum"),
    discount_amount = ("total" , lambda x : abs(x[x<0].sum())),
    items_qty = ("qty_ordered" , lambda x : data_clean.loc[x.index , "is_real_product_row"].sum())


).reset_index()


In [ ]:
# -------------------------------------------------
# 2) Product Variant Features
# -------------------------------------------------

product_rows = data_clean[data_clean["is_real_product_row"]].copy()
product_rows["product_variant_clean"]  = product_rows["product_variant_clean"] = (
    product_rows["product_variant"].astype(str).str.strip()
)

product_features= product_rows.pivot_table(
    index = "order_id",
    columns = "product_variant_clean",
    values="qty_ordered",
    aggfunc="sum",
    fill_value= 0
)


product_features.columns=[
    safe_column_name(col , "qty_ordered")
    for col in product_features.columns
]

product_features = product_features.reset_index()

product_features_columns = [
   col for col in product_features.columns
   if col != "order_id"
]






In [ ]:
# -------------------------------------------------
# 3) Financial Features
# -------------------------------------------------

financial_rows=data_clean[data_clean["financial_category"].notna()].copy()

financial_amount_features = financial_rows.pivot_table(
    index = "order_id",
    columns= "financial_category",
    values="financial_amount_for_feature",
    aggfunc="sum",
    fill_value=0
)

financial_amount_features.columns=[
    safe_column_name(col , "financial_amount")
    for col in financial_amount_features.columns
]

financial_amount_features = financial_amount_features.reset_index()

financial_amount_columns = [
    col for col in financial_amount_features.columns
    if col != "order_id"
]

financial_has_features = pd.crosstab(
    financial_rows["order_id"],
    financial_rows["financial_category"]
)

financial_has_features = (financial_has_features > 0).astype(int)

financial_has_columns = [
    col for col in financial_has_features.columns
    if col != "order_id"
]


financial_feature_columns = financial_amount_columns + financial_has_columns



In [ ]:
# -------------------------------------------------
# 4) Service Features
# -------------------------------------------------

service_rows = data_clean[
    (data_clean["service"].notna() ) &
     (data_clean["service"].astype(str).str.strip() != "")
     ].copy()


service_rows["service_clean"] = (service_rows["service"].astype(str).str.strip())

service_features=pd.crosstab(
    service_rows["order_id"],
    service_rows["service_clean"]
)


service_features = (service_features > 0).astype(int)

service_features.columns=[
    safe_column_name(col , "has_service")
    for col in service_features.columns
]

service_features = service_features.reset_index()


service_features_columns = [
    col for col in service_features.columns
    if col != "order_id"
]


In [ ]:
# -------------------------------------------------
# 5) Merge product + financial + service features مع order_level
# -------------------------------------------------

order_level = order_level.merge(
    product_features,
    on = "order_id",
    how = "left"
)


order_level = order_level.merge(
    financial_amount_features,
    on = "order_id",
    how = "left"
)

order_level = order_level.merge(
    financial_has_features,
    on = "order_id",
    how = "left"
)

order_level = order_level.merge(
    service_features,
    on = "order_id",
    how = "left"
)

if len(product_features_columns) > 0 :
  order_level[product_features_columns] = order_level[product_features_columns].fillna(0)

if len(financial_amount_columns) > 0 :
  order_level[financial_feature_columns] = order_level[financial_feature_columns].fillna(0)

if len(service_features_columns) > 0 :
  order_level[service_features_columns] = order_level[service_features_columns].fillna(0)


if len(product_features_columns) > 0:
    order_level[product_features_columns] = order_level[product_features_columns].astype(float)

if len(financial_amount_columns) > 0:
    order_level[financial_amount_columns] = order_level[financial_amount_columns].astype(float)

if len(financial_has_columns) > 0:
    order_level[financial_has_columns] = order_level[financial_has_columns].astype(int)

if len(service_features_columns) > 0:
    order_level[service_features_columns] = order_level[service_features_columns].astype(int)


In [ ]:
# -------------------------------------------------
# 6) فحوصات
# -------------------------------------------------

print("Original rows:", len(data_clean))
print("Order-level rows:", len(order_level))

print("\nProduct feature columns count:", len(product_features_columns))
print("Financial feature columns count:", len(financial_feature_columns))
print("Service feature columns count:", len(service_features_columns))

print("\nFirst 10 product feature columns:")
print(product_features_columns[:10])

print("\nFinancial feature columns:")
print(financial_feature_columns)

print("\nService feature columns:")
print(service_features_columns)

print("\nOrder-level date range:")
print(order_level["order_date"].min(), "to", order_level["order_date"].max())

print("\nBag status distribution:")
print(order_level["bag"].value_counts(dropna=False))

print("\nSubscription status distribution:")
print(order_level["has_active_subscription"].value_counts(dropna=False))

print("\nService combinations examples:")
print(order_level["service"].value_counts().head(20))

print("\nFinancial items examples:")
print(order_level["financial_items"].value_counts().head(20))

print("\nProduct variants examples:")
display(order_level[[
    "order_id",
    "customer_id",
    "customer_phone",
    "has_active_subscription",
    "product_variants",
    "financial_items",
    "service",
    "bag",
    "items_qty",
    "order_revenue",
    "discount_amount"
]].head(20))

display(order_level.head())


## 3. Target Engineering

Create the supervised target: whether the customer returns within 30 days after an order.


In [ ]:
# =========================
# Block 6: Create target variable
# =========================

order_level = order_level.sort_values(
    ["customer_id" , "order_date"]
).reset_index(drop = True)


order_level["next_order_date"] = (
    order_level.groupby("customer_id")["order_date"].shift(-1)
)


order_level["days_to_next_order"]=(
    order_level["next_order_date"] - order_level["order_date"]
).dt.days


order_level["returned_within_30_days"]= (
    (order_level["days_to_next_order"].notna()) &
    (order_level["days_to_next_order"] <= 30)
).astype(int)



max_data_date = order_level['order_date'].max()

label_cutoff_date = max_data_date - pd.Timedelta(days = 30)

order_level["label_available"]= (
    order_level["order_date"] <= label_cutoff_date
)


print("Max data date:", max_data_date)
print("Label cutoff date:", label_cutoff_date)
print("\nRows total:", len(order_level))
print(
    "Rows with available label:",
    order_level['label_available'].sum()
)

print(
    "Rows without available label:",
    (~order_level['label_available']).sum()
)

print("\nTarget distribution - all rows:")
print(
    order_level['returned_within_30_days'].value_counts()
)

print("\nTarget distribution percentage - all rows:")
print(
    order_level['returned_within_30_days']
    .value_counts(normalize=True) * 100
)

print("\nTarget distribution - label available only:")
print(order_level.loc[order_level["label_available"] , "returned_within_30_days"].value_counts())

print("\nTarget distribution percentage - label available only:")
print(order_level.loc[order_level["label_available"] , "returned_within_30_days"].value_counts(normalize =True) * 100)

display(order_level[[
    'customer_id',
    'order_id',
    'order_date',
    'next_order_date',
    'days_to_next_order',
    'returned_within_30_days',
    'label_available'
]].head(20))



In [ ]:
# =========================
# Block 7: Keep only rows with available labels for training
# =========================
train_base = order_level[order_level["label_available"]].copy()

print("Orders before label filtering:", len(order_level))
print("Orders after label filtering:", len(train_base))
print("Orders removed:", len(order_level) - len(train_base))

print("\nDate range used for training:")
print(train_base['order_date'].min(), "to", train_base['order_date'].max())

print("\nTarget distribution after label filtering:")
print(train_base['returned_within_30_days'].value_counts())

print("\nTarget distribution percentage after label filtering:")
print(train_base['returned_within_30_days'].value_counts(normalize=True) * 100)


## 4. Customer History Features

Build leakage-aware historical features using only information available before each prediction point.


In [ ]:
# =========================
# Block 8: Create customer history features
# =========================

import numpy as np

order_features = order_level.sort_values(
    [ "customer_id" , "order_date"]
).copy()


order_features["previous_orders_count"] = (
    order_features.groupby("customer_id").cumcount()
)


order_features["previous_total_spend"]= (
    order_features.groupby("customer_id")["order_revenue"]
    .cumsum()
    .groupby(order_features["customer_id"])
    .shift(1)
    .fillna(0)
)

order_features['previous_total_spend'] = (
    order_features["previous_total_spend"]
    .clip(lower=0)
)

order_features["previous_avg_order_value"] = np.where(
    order_features["previous_orders_count"] > 0,
    order_features["previous_total_spend"] / order_features["previous_orders_count"],
    0
)

order_features['previous_total_discount'] = (
    order_features.groupby('customer_id')['discount_amount']
    .cumsum()
    .groupby(order_features['customer_id'])
    .shift(1)
    .fillna(0)
)


order_features['previous_total_discount'] = (
    order_features['previous_total_discount']
    .clip(lower=0)
)


order_features['previous_discount_ratio'] = np.where(
    order_features['previous_total_spend'] > 0,
    order_features['previous_total_discount'] / order_features['previous_total_spend'],
    0
)

order_features["previous_discount_ratio"] = (
    pd.Series(order_features["previous_discount_ratio"])
    .replace([np.inf , -np.inf] , 0)
    .fillna(0)
    .clip(lower = 0  , upper = 1)
)


order_features["previos_order_date"]=(
    order_features.groupby("customer_id")["order_date"].shift(1)
)

order_features["days_since_previous_order"]= (
    order_features["order_date"] - order_features["previos_order_date"]
).dt.days.fillna(999)

order_features['is_first_order'] = (
    order_features['previous_orders_count'] == 0
).astype(int)



print("Order features rows:", len(order_features))

print("\nDate range:")
print(order_features['order_date'].min(), "to", order_features['order_date'].max())

print("\nLabel availability distribution:")
print(order_features['label_available'].value_counts())

print("\nHistory feature checks:")
print(order_features[[
    'previous_orders_count',
    'previous_total_spend',
    'previous_avg_order_value',
    'previous_total_discount',
    'previous_discount_ratio',
    'days_since_previous_order',
    'is_first_order'
]].describe())

display(order_features[[
    'customer_id',
    'order_id',
    'order_date',
    'order_revenue',
    'previous_orders_count',
    'previous_total_spend',
    'previous_avg_order_value',
    'previous_total_discount',
    'previous_discount_ratio',
    'days_since_previous_order',
    'is_first_order',
    'returned_within_30_days',
    'label_available'
]].head(30))


## 5. Training Dataset Preparation

Select model features and prepare categorical/numeric inputs for training.


In [ ]:
# =========================
# Block 9: Prepare final training dataset
# =========================

train_base_v2 = order_features[
    order_features["label_available"] == True
].copy()

base_order_features = [
    'order_revenue',
    'discount_amount',
    'items_qty',
    'order_category',
    'express',
    'bag',
    'residency_type',
    'laundry_type'
]

customer_history_features = [
    'previous_orders_count',
    'previous_total_spend',
    'previous_avg_order_value',
    'previous_total_discount',
    'previous_discount_ratio',
    'days_since_previous_order',
    'is_first_order'
]

feature_columns_v2 = (
    base_order_features
    + customer_history_features
    + product_features_columns
    + financial_feature_columns
    + service_features_columns
)

target_column = 'returned_within_30_days'

missing_feature_columns = [
    col for col in feature_columns_v2
    if col not in train_base_v2.columns
]

if missing_feature_columns:
  print("Missing feature columns:")
  print(missing_feature_columns)
  raise ValueError("Some feature columns are missing from train_base_v2")


model_data_v2 = train_base_v2[
    feature_columns_v2 + [target_column]
].copy()


categorical_columns = [
    'order_category',
    'bag',
    'residency_type',
    'laundry_type'
]


for col in categorical_columns:
  if col in model_data_v2.columns:
    model_data_v2[col]=(
        model_data_v2[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

  model_data_v2[col]=model_data_v2[col].replace(
      ['nan', 'None', 'NaN', '', '<NA>'],
      'Unknown'
  )

numeric_columns_to_fill = (
    ['order_revenue', 'discount_amount', 'items_qty', 'express']
    + customer_history_features
    + product_features_columns
    + financial_feature_columns
    + service_features_columns
)

numeric_columns_to_fill = [
    col for col in numeric_columns_to_fill
    if col in model_data_v2.columns
]

model_data_v2[numeric_columns_to_fill] = (
    model_data_v2[numeric_columns_to_fill]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

display(model_data_v2.head())

print("Train base v2 rows:", len(train_base_v2))
print("Model data v2 shape:", model_data_v2.shape)

print("\nBase order features count:", len(base_order_features))
print("Customer history features count:", len(customer_history_features))
print("Product feature columns count:", len(product_features_columns))
print("Financial feature columns count:", len(financial_feature_columns))
print("Service feature columns count:", len(service_features_columns))
print("Total feature columns count:", len(feature_columns_v2))

print("\nMissing values:")
print(model_data_v2.isna().sum().sort_values(ascending=False).head(30))

print("\nCategorical values check:")
for col in categorical_columns:
    print(f"\n{col}:")
    print(model_data_v2[col].value_counts(dropna=False).head(20))

print("\nTarget distribution:")
print(model_data_v2[target_column].value_counts())

print("\nTarget distribution percentage:")
print(model_data_v2[target_column].value_counts(normalize=True) * 100)


In [ ]:
# =========================
# Block 10: Encode categorical columns
# =========================

model_ready_v2 = model_data_v2.copy()

categorical_columns = [
    'order_category',
    'bag',
    'residency_type',
    'laundry_type'
]

model_ready_v2[col] = (
    model_ready_v2[col]
    .fillna('Unknown')
    .astype(str)
    .str.strip()
    .replace(['nan', 'None', 'NaN', '', '<NA>'], 'Unknown')
)


model_ready_v2['express'] = (
    pd.to_numeric(model_ready_v2['express'], errors='coerce')
    .fillna(0)
    .astype(int)
)


numeric_feature_columns = (
    ['order_revenue', 'discount_amount', 'items_qty', 'express']
    + customer_history_features
    + product_features_columns
    + financial_feature_columns
    + service_features_columns
)


numeric_feature_columns = [
    col for col in numeric_feature_columns
    if col in model_ready_v2.columns
]

model_ready_v2[numeric_feature_columns] = (
    model_ready_v2[numeric_feature_columns]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


model_ready_v2[product_features_columns] = model_ready_v2[product_features_columns].astype(float)
model_ready_v2[financial_feature_columns] = model_ready_v2[financial_feature_columns].astype(float)
model_ready_v2[service_features_columns] = model_ready_v2[service_features_columns].astype(int)

X_v2 = model_ready_v2.drop(columns=[target_column])
y_v2 = model_ready_v2[target_column].astype(int)



X_encoded_v2 = pd.get_dummies(
    X_v2 ,
    columns = categorical_columns ,
    drop_first= True
)

print("Original X_v2 shape:", X_v2.shape)
print("Encoded X_v2 shape:", X_encoded_v2.shape)
print("Target y_v2 shape:", y_v2.shape)

print("\nMissing values after filling:")
print(model_ready_v2.isna().sum().sort_values(ascending=False).head(30))

print("\nEncoded missing values:")
print(X_encoded_v2.isna().sum().sum())

print("\nEncoded columns count:", len(X_encoded_v2.columns))

print("\nTarget distribution:")
print(y_v2.value_counts())
print(y_v2.value_counts(normalize=True) * 100)

display(X_encoded_v2.head())


## 6. Random Forest Training with Time-Based Validation

Use a chronological train/test split rather than a random split to better simulate future production scoring.


In [ ]:
# =========================
# Block 11: Train Random Forest model with time-based validation
# =========================


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score
)


In [ ]:
# =========================
# 1) Time-based split
# =========================
train_mask_v2 =train_base_v2["order_date"] < pd.Timestamp('2026-05-01')

test_mask_v2 = (
    (train_base_v2['order_date'] >= pd.Timestamp('2026-05-01'))&
    (train_base_v2['order_date'] <= pd.Timestamp('2026-05-31 23:59:59'))
)


X_train_v2 = X_encoded_v2.loc[train_mask_v2]
y_train_v2 = y_v2.loc[train_mask_v2]

X_test_v2 = X_encoded_v2.loc[test_mask_v2]
y_test_v2 = y_v2.loc[test_mask_v2]

print("Train date range:")
print(train_base_v2.loc[train_mask_v2, 'order_date'].min(), "to", train_base_v2.loc[train_mask_v2, 'order_date'].max())

print("\nTest date range:")
print(train_base_v2.loc[test_mask_v2, 'order_date'].min(), "to", train_base_v2.loc[test_mask_v2, 'order_date'].max())

print("\nTrain rows:", X_train_v2.shape[0])
print("Test rows:", X_test_v2.shape[0])

print("\nTrain target distribution:")
print(y_train_v2.value_counts())
print(y_train_v2.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test_v2.value_counts())
print(y_test_v2.value_counts(normalize=True) * 100)

model_feature_columns_v2 = X_encoded_v2.columns.tolist()


In [ ]:
# =========================
# 2) Train Random Forest
# =========================
rf_model_v2 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced_subsample',
    n_jobs=-1,
    max_features='sqrt',
    min_samples_split=4,
    min_samples_leaf=2
)

rf_model_v2.fit(X_train_v2,y_train_v2)




In [ ]:
# =========================
# 3) Predict on May test set
# =========================
y_pred_rf_v2 = rf_model_v2.predict(X_test_v2)
y_proba_rf_v2 = rf_model_v2.predict_proba(X_test_v2)[:, 1]



## 7. Model Evaluation

The original model achieved approximately:

- **Accuracy:** 81.7%
- **ROC-AUC:** 0.881
- **Precision:** 0.847
- **Recall:** 0.887
- **F1 Score:** 0.867

Only aggregate performance metrics are documented here; customer-level outputs were removed.


In [ ]:
# =========================
# 4) Evaluation
# =========================
print( "\nRandom Forest v2 Accuracy:", accuracy_score(y_test_v2 , y_pred_rf_v2))
print("ROC AUC:", roc_auc_score(y_test_v2, y_proba_rf_v2))

print("Precision:", precision_score(y_test_v2, y_pred_rf_v2, zero_division=0 ))
print("Recall:", recall_score(y_test_v2, y_pred_rf_v2, zero_division=0))
print("F1 Score:", f1_score(y_test_v2, y_pred_rf_v2, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v2, y_pred_rf_v2))

print("\nClassification Report:")
print(classification_report(y_test_v2, y_pred_rf_v2, zero_division=0))


## 8. Feature Importance

Inspect the most influential customer-history, product, financial, and service features.


In [ ]:
# =========================
# Block 12: Feature importance
# =========================

feature_importance_v2 = pd.DataFrame({
    'feature': model_feature_columns_v2,
    'importance': rf_model_v2.feature_importances_
})

feature_importance_v2 = feature_importance_v2.sort_values(
    by='importance',
    ascending=False
).reset_index(drop=True)

print("Feature importance rows:", len(feature_importance_v2))
print("Total importance:", feature_importance_v2['importance'].sum())

print("\nTop 30 important features:")
display(feature_importance_v2.head(30))


In [ ]:
# =========================
# Group feature importance by type
# =========================

def feature_group(feature_name):
  if feature_name in product_features_columns:
    return "Product Features"

  elif feature_name in financial_feature_columns:
      return 'Financial Features'

  elif feature_name in service_features_columns:
      return 'Service Features'


  elif feature_name.startswith('previous_') or feature_name in ['days_since_previous_order','is_first_order']:
    return 'Customer History Features'

  elif feature_name.startswith('order_category_'):
        return 'Order Category Features'

  elif feature_name.startswith('residency_type_'):
      return 'Residency Type Features'

  elif feature_name.startswith('laundry_type_'):
      return 'Laundry Type Features'

  elif feature_name.startswith('bag_'):
      return 'Bag Features'

  elif feature_name in [
      'order_revenue',
      'discount_amount',
      'items_qty',
      'express'
  ]:
      return 'Current Order Features'

  else:
      return 'Other Features'


feature_importance_v2["feature_group"] = (
    feature_importance_v2['feature'].apply(feature_group)
)

group_importance = (
    feature_importance_v2.groupby('feature_group')['importance'].sum().reset_index().sort_values(by= "importance" , ascending=False)
)

print("\nFeature importance by group:")
display(group_importance)


In [ ]:
# =========================
# Top product features only
# =========================

product_importance = feature_importance_v2[feature_importance_v2['feature'].isin(product_features_columns) ].copy()


print("\nTop 30 product features:")
display(product_importance.head(30))



In [ ]:
# =========================
# Financial features only
# =========================

financial_importance = feature_importance_v2[
    feature_importance_v2['feature'].isin(financial_feature_columns)
  ].copy()

print("\nFinancial features importance:")
display(financial_importance)




In [ ]:
# =========================
# Service features only
# =========================

service_importance = feature_importance_v2[
    feature_importance_v2['feature'].isin(service_features_columns)
].copy()

print("\nService features importance:")
display(service_importance)


## 9. Customer Scoring & Risk Segmentation

Generate return probabilities and convert them into business-friendly customer segments.


In [ ]:
# =========================
# Block 13: Create scoring table for May test data
# =========================
y_proba_v2 = y_proba_rf_v2
test_indices = X_test_v2.index

print("Test rows:", len(test_indices))
print("Predictions count:", len(y_pred_rf_v2))
print("Probabilities count:", len(y_proba_v2))

if len(test_indices) != len(y_pred_rf_v2):
  raise ValueError("Mismatch between test rows and predictions count")



In [ ]:
# الأعمدة المفهومة للمراجعة
scoring_columns = [
    'customer_id',
    'customer_phone',
    'has_active_subscription',
    'order_id',
    'order_date',
    'order_revenue',
    'discount_amount',
    'items_qty',
    'order_category',
    'service',
    'product_variants',
    'financial_items',
    'bag',
    'residency_type',
    'laundry_type',
    'previous_orders_count',
    'previous_total_spend',
    'previous_avg_order_value',
    'previous_total_discount',
    'previous_discount_ratio',
    'days_since_previous_order',
    'is_first_order',
    'returned_within_30_days'
]

scoring_columns = [
    col for col in scoring_columns
    if col in train_base_v2.columns
]

scored_customers = train_base_v2.loc[
    test_indices ,
    scoring_columns
].copy()


scored_customers['predicted_returned_within_30_days'] = y_pred_rf_v2
scored_customers['return_probability'] = y_proba_v2


In [ ]:
# تحويل الاحتمال إلى شريحة تسويقية
def risk_segment(prob):
    if prob < 0.40:
        return 'High Risk - يحتاج تدخل'
    elif prob < 0.70:
        return 'Medium Risk - يحتاج تذكير'
    elif prob < 0.90:
        return 'Likely Return - لا يحتاج خصم قوي'
    else:
        return 'Very Likely Return - احتمال رجوع مرتفع جدًا'


scored_customers['risk_segment'] = (
    scored_customers['return_probability']
    .apply(risk_segment)
)


scored_customers = scored_customers.sort_values(
    by = 'return_probability',
    ascending = True

).reset_index(drop=True)

print("Scored customers shape:", scored_customers.shape)

print("\nRisk segment distribution:")
print(scored_customers['risk_segment'].value_counts())

print("\nRisk segment percentage:")
print(scored_customers['risk_segment'].value_counts(normalize = True) * 100)

print("\nPrediction vs Actual:")
print(pd.crosstab(
    scored_customers['returned_within_30_days'],
    scored_customers['predicted_returned_within_30_days'],
    rownames=['Actual'],
    colnames=['Predicted']

))

display(scored_customers.head(20))

print("\nPrediction vs Actual sample:")
display(scored_customers[[
    'customer_id',
    'customer_phone',
    'has_active_subscription',
    'order_id',
    'order_date',
    'product_variants',
    'financial_items',
    'service',
    'returned_within_30_days',
    'predicted_returned_within_30_days',
    'return_probability',
    'risk_segment'
]].head(30))


In [ ]:
# =========================
# Block 14: Export May test scored customers
# =========================

export_results = scored_customers[[
    'customer_id',
    'customer_phone',
    'has_active_subscription',

    'order_id',
    'order_date',
    'order_revenue',
    'discount_amount',
    'items_qty',

    'order_category',
    'service',
    'product_variants',
    'financial_items',
    'bag',
    'residency_type',
    'laundry_type',

    'previous_orders_count',
    'previous_total_spend',
    'previous_avg_order_value',
    'previous_total_discount',
    'previous_discount_ratio',
    'days_since_previous_order',
    'is_first_order',

    'returned_within_30_days',
    'predicted_returned_within_30_days',
    'return_probability',
    'risk_segment'
]].copy()

export_results = export_results.rename(columns={
    'order_id': 'last_order_id',
    'order_date': 'last_order_date',
    'order_revenue': 'last_order_revenue',
    'discount_amount': 'last_order_discount_amount',
    'items_qty': 'last_order_items_qty',
    'product_variants': 'last_order_product_variants',
    'financial_items': 'last_order_financial_items',
    'service': 'last_order_services'
})

export_results = export_results.sort_values(
    'return_probability',
    ascending=True
).reset_index(drop=True)


export_results.to_csv(
    'may_test_customer_return_predictions.csv',
    index =False,
    encoding='utf-8-sig'
)

print("File saved: may_test_customer_return_predictions.csv")
print("Rows exported:", len(export_results))

print("\nRisk segment distribution:")
print(export_results['risk_segment'].value_counts())

print("\nPrediction vs Actual:")
print(pd.crosstab(
    export_results['returned_within_30_days'],
    export_results['predicted_returned_within_30_days'],
    rownames=['Actual'],
    colnames=['Predicted']
))

display(export_results.head(20))


In [ ]:
# =========================
# Block 15: Validate May Test Predictions
# =========================
# =========================
# 1) Actual return rate by risk segment
# =========================

segment_validation = (
    scored_customers
    .groupby('risk_segment')
    .agg(
        customers_count=('customer_id', 'count'),
        avg_predicted_probability=('return_probability', 'mean'),
        actual_return_rate=('returned_within_30_days', 'mean'),
        predicted_return_rate=('predicted_returned_within_30_days', 'mean')
    )
    .reset_index()
)

segment_validation['actual_return_rate_pct'] = (
    segment_validation['actual_return_rate'] * 100
)

segment_validation['avg_predicted_probability_pct'] = (
    segment_validation['avg_predicted_probability'] * 100
)

segment_validation = segment_validation.sort_values(
    'avg_predicted_probability',
    ascending=True
)

print("\nValidation by Risk Segment:")
display(segment_validation)



In [ ]:
# =========================
# 2) Probability bins validation
# =========================

scored_customers ['probability_bin'] = pd.cut(
    scored_customers['return_probability'],
    bins = [0 , 0.2 , 0.4 , 0.6 , 0.8 , 1.0],

    labels= [
        '0-20%',
        '20-40%',
        '40-60%',
        '60-80%',
        '80-100%'
    ],
    include_lowest=True
)

probability_validation = (
  scored_customers
  .groupby('probability_bin', observed=False)
  .agg(
      customers_count=('customer_id', 'count'),
      avg_predicted_probability=('return_probability', 'mean'),
      actual_return_rate=('returned_within_30_days', 'mean')
  )
  .reset_index()
)

probability_validation['avg_predicted_probability_pct'] = (
  probability_validation['avg_predicted_probability'] * 100
)

probability_validation['actual_return_rate_pct'] = (
  probability_validation['actual_return_rate'] * 100
)

print("\nValidation by Probability Bin:")
display(probability_validation)


In [ ]:
# =========================
# 3) Correct vs wrong predictions
# =========================

scored_customers['prediction_correct'] = (
    scored_customers['returned_within_30_days'] ==
    scored_customers['predicted_returned_within_30_days']
)

print("\nCorrect prediction distribution:")
print(scored_customers['prediction_correct'].value_counts())

print("\nCorrect prediction percentage:")
print(scored_customers['prediction_correct'].value_counts(normalize=True) * 100)

print("\nCorrect prediction examples:")
display(
    scored_customers[
        scored_customers['prediction_correct'] == True
    ][[
        'customer_id',
        'order_id',
        'order_date',
        'returned_within_30_days',
        'predicted_returned_within_30_days',
        'return_probability',
        'risk_segment'
    ]].head(20)
)


print("\nWrong prediction examples:")
display(
    scored_customers[
        scored_customers['prediction_correct'] == False
    ][[
        'customer_id',
        'order_id',
        'order_date',
        'returned_within_30_days',
        'predicted_returned_within_30_days',
        'return_probability',
        'risk_segment'
    ]].head(20)
)


## 10. Final Model & Latest-Customer Scoring

Retrain on the full labeled period and score each customer's latest known state.


In [ ]:
# =========================
# Block 16: Train Final Model on Jan-May
# =========================

X_final_train_v2 = X_encoded_v2.copy()
y_final_train_v2 = y_v2.copy()


print("Final training rows:", X_final_train_v2.shape[0])
print("Final training columns:", X_final_train_v2.shape[1])

print("\nFinal training target distribution:")
print(y_final_train_v2.value_counts())

print("\nFinal training target distribution percentage:")
print(y_final_train_v2.value_counts(normalize=True) * 100)

final_model_feature_columns_v2 = X_final_train_v2.columns.tolist()



In [ ]:
# =========================
# تدريب الموديل النهائي
# =========================

rf_final_model_v2 = RandomForestClassifier(
    n_estimators =300,
    random_state = 42,
    class_weight = 'balanced_subsample',
    n_jobs = -1,
    max_features = 'sqrt',
    min_samples_split = 4,
    min_samples_leaf = 2
)

rf_final_model_v2.fit(X_final_train_v2 , y_final_train_v2)

print("\nFinal Random Forest model trained successfully.")
print("Features used:", len(final_model_feature_columns_v2))


In [ ]:
# =========================
# Block 16.1: Score latest customer status as of end of June
# =========================

latest_order = (
    order_features
    .sort_values(['customer_id' , 'order_date'])
    .groupby('customer_id')
    .tail(1)
).copy()

print("Latest Cusrtomer Rows" , len(latest_order))
print("Latest order date range : ")
print(latest_order['order_date'].min(), "to", latest_order['order_date'].max())


In [ ]:
# نجهز نفس أعمدة التدريب
latest_model_data = latest_order[feature_columns_v2].copy()

categorical_columns = [
    'order_category',
    'bag',
    'residency_type',
    'laundry_type'
]


for col in categorical_columns:
  latest_model_data[col] = (
      latest_model_data[col]
      .fillna('Unknown')
      .astype(str)
      .str.strip()
      .replace(['nan', 'None', 'NaN', '', '<NA>'], 'Unknown')
  )


latest_model_data['express'] = (
    pd.to_numeric(latest_model_data['express'] , errors = 'coerce')
    .fillna(0)
    .astype(int)
)

numeric_feature_columns = (
    [
        'order_revenue',
        'discount_amount',
        'items_qty',
        'express'
    ]
    + customer_history_features
    + product_features_columns
    + financial_feature_columns
    + service_features_columns
)

numeric_feature_columns =[
    col for col in numeric_feature_columns
    if col in latest_model_data.columns
]


latest_model_data[numeric_feature_columns] = (
    latest_model_data[numeric_feature_columns]
    .replace([np.inf , -np.inf], 0 )
    .fillna(0)
)


latest_encoded = pd.get_dummies(
    latest_model_data ,
    columns = categorical_columns ,
    drop_first= True
)


latest_encoded =  latest_encoded.reindex(
    columns = final_model_feature_columns_v2 ,
    fill_value = 0
)

print("Latest encoded shape:", latest_encoded.shape)
print("Final model features:", len(final_model_feature_columns_v2))
print("Missing values in latest encoded:", latest_encoded.isna().sum().sum())



In [ ]:
# توقع احتمالية الرجوع
latest_return_probability = rf_final_model_v2.predict_proba(latest_encoded)[: , 1]
latest_return_prediction = rf_final_model_v2.predict(latest_encoded)


In [ ]:
# جدول التوقعات النهائي قبل Marketing Layer
latest_customer_predictions = latest_order[[
    'customer_id',
    'customer_phone',
    'has_active_subscription',
    'order_id',
    'order_date',
    'order_revenue',
    'discount_amount',
    'items_qty',
    'order_category',
    'service',
    'product_variants',
    'financial_items',
    'bag',
    'residency_type',
    'laundry_type',
    'previous_orders_count',
    'previous_total_spend',
    'previous_avg_order_value',
    'previous_total_discount',
    'previous_discount_ratio',
    'days_since_previous_order',
    'is_first_order'
]].copy()

latest_customer_predictions['predicted_returned_within_30_days'] = latest_return_prediction
latest_customer_predictions['return_probability'] = latest_return_probability



In [ ]:
# نفس تقسيم الشرائح
def risk_segment(prob):
    if prob < 0.40:
        return 'High Risk - يحتاج تدخل'
    elif prob < 0.70:
        return 'Medium Risk - يحتاج تذكير'
    elif prob < 0.90:
        return 'Likely Return - لا يحتاج خصم قوي'
    else:
        return 'Very Likely Return - ولاء عالي'

latest_customer_predictions['risk_segment'] = (
    latest_customer_predictions['return_probability'].apply(risk_segment)
)

# ترتيب من الأقل احتمال رجوع إلى الأعلى
latest_customer_predictions = latest_customer_predictions.sort_values(
    'return_probability',
    ascending=True
).reset_index(drop=True)


In [ ]:
print("Latest customer predictions shape:", latest_customer_predictions.shape)

print("\nRisk segment distribution:")
print(latest_customer_predictions['risk_segment'].value_counts())

print("\nRisk segment percentage:")
print(latest_customer_predictions['risk_segment'].value_counts(normalize=True) * 100)

display(latest_customer_predictions.head(30))


In [ ]:
# =========================
# Block 16.2: Sanity check latest customer predictions
# =========================

print("Latest customer predictions shape:")
print(latest_customer_predictions.shape)

print("\nMissing phone count:")
print(latest_customer_predictions['customer_phone'].isna().sum())

print("\nActive subscription distribution:")
print(latest_customer_predictions['has_active_subscription'].value_counts(dropna=False))

print("\nLatest order month distribution:")
print(
    latest_customer_predictions['order_date']
    .dt.to_period('M')
    .value_counts()
    .sort_index()
)

print("\nRisk segment distribution:")
print(latest_customer_predictions['risk_segment'].value_counts())

print("\nRisk segment percentage:")
print(latest_customer_predictions['risk_segment'].value_counts(normalize=True) * 100)

display(latest_customer_predictions.head(20))


# 🎯 Marketing Targeting Layer

Build customer value, frequency, inactivity, promotional behavior, and subscription/package recommendation layers on top of the ML predictions.


In [ ]:
# =========================
# Block 17: Final Marketing Recommendation Layer
# =========================

final_marketing = latest_customer_predictions.copy()

print("Marketing base rows:", len(final_marketing))
print("Marketing base columns:", len(final_marketing.columns))


In [ ]:
# =========================
# 1) Prepare numeric columns
# =========================

numeric_cols = [
    'order_revenue',
    'discount_amount',
    'items_qty',
    'previous_orders_count',
    'previous_total_spend',
    'previous_avg_order_value',
    'previous_total_discount',
    'previous_discount_ratio',
    'days_since_previous_order',
    'return_probability'
]


for col in numeric_cols:
  if col in final_marketing:
    final_marketing[col] = pd.to_numeric(
        final_marketing[col] , errors = 'coerce'
    ).replace([np.inf , -np.inf] , 0 ).fillna(0)


if 'customer_phone' in final_marketing:
  final_marketing['customer_phone'] = (
      final_marketing['customer_phone']
      .astype(str)
      .str.replace(r'\.0$' , '' , regex  =True)
      .str.strip()
      .replace(['nan', 'None', '<NA>', ''], pd.NA)
  )


if 'has_active_subscription' in final_marketing.columns:
    final_marketing['has_active_subscription'] = (
        final_marketing['has_active_subscription']
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(['true', '1', 'yes'])
    )
else:
    final_marketing['has_active_subscription'] = False



In [ ]:
# =========================
# 2) Customer history status
# =========================

def customer_history_status (row):
  orders = row['previous_orders_count']
  days = row['days_since_previous_order']

  if orders <= 0 or days >= 999:
    return 'New Customer / No Previous Order'

  return 'Returning Customer'

final_marketing['Customer History Status']  = final_marketing.apply(
    customer_history_status , axis = 1
)


def clean_days_since_previous(row):
  orders = row['previous_orders_count']
  days = row['days_since_previous_order']

  if orders <= 0 or days >= 999:
    return 'No Previous Order'

  return int(days)


final_marketing['Days Since Previous Order - Clean'] = final_marketing.apply(
    clean_days_since_previous , axis = 1
)


In [ ]:
# =========================
# 3) Customer value segment
# =========================

final_marketing['Total Orders To Date'] = (
    final_marketing['previous_orders_count'] + 1
)

final_marketing['Total Spend To Date'] = (
    final_marketing['previous_total_spend']
    + final_marketing['order_revenue']
)

final_marketing['Customer AOV To Date'] = (
    final_marketing['Total Spend To Date']
    / final_marketing['Total Orders To Date']
)

display(
    final_marketing[[
        'customer_id',
        'previous_orders_count',
        'order_revenue',
        'previous_total_spend',
        'Total Orders To Date',
        'Total Spend To Date',
        'Customer AOV To Date'
    ]].head(10)
)


In [ ]:
# تجهيز بيانات نشاط الحسابات من كل الطلبات

account_activity_base = order_features[[
    'customer_id',
    'order_id',
    'order_date'
]].copy()

account_activity_base['order_date'] = pd.to_datetime(
    account_activity_base['order_date'],
    errors='coerce'
)

account_activity_base['order_day'] = (
    account_activity_base['order_date'].dt.date
)

daily_orders = (
    account_activity_base
    .groupby([
        'customer_id',
        'order_day'
    ])['order_id']
    .nunique()
    .reset_index(
        name='orders_same_day'
    )
)

account_activity_summary = (
    daily_orders
    .groupby('customer_id')
    .agg(
        total_orders=('orders_same_day', 'sum'),
        active_days=('order_day', 'nunique'),
        average_orders_per_active_day=('orders_same_day', 'mean'),
        maximum_orders_in_one_day=('orders_same_day', 'max')
    )
    .reset_index()
)

maximum_daily_orders_distribution = (
    account_activity_summary[
        'maximum_orders_in_one_day'
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        'maximum_orders_in_one_day'
    )
    .reset_index(
        name='customer_count'
    )
)

display(maximum_daily_orders_distribution)


suspected_non_individual_accounts = (
    account_activity_summary[
        account_activity_summary[
            'maximum_orders_in_one_day'
        ] >= 5
    ]
    .sort_values(
        'maximum_orders_in_one_day',
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Suspected non-individual accounts:",
    len(suspected_non_individual_accounts)
)

display(suspected_non_individual_accounts)


In [ ]:
suspected_non_individual_ids = (
    suspected_non_individual_accounts['customer_id']
    .unique()
)

final_marketing['Suspected Non-Individual Account'] = (
    final_marketing['customer_id']
    .isin(suspected_non_individual_ids)
)


print(
    final_marketing[
        'Suspected Non-Individual Account'
    ].value_counts()
)


In [ ]:
individual_value_base = final_marketing[
    final_marketing['Suspected Non-Individual Account'] == False
].copy()

print("All customers:", len(final_marketing))
print("Customers used for value analysis:", len(individual_value_base))


In [ ]:
returning_value_base = individual_value_base[
    individual_value_base['Total Orders To Date'] > 1
].copy()

new_customer_value_base = individual_value_base[
    individual_value_base['Total Orders To Date'] == 1
].copy()


In [ ]:
pd.set_option(
    'display.float_format',
    lambda x: f'{x:,.2f}'
)

display(
    individual_value_base[[
        'Total Orders To Date',
        'Total Spend To Date',
        'Customer AOV To Date'
    ]].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)


In [ ]:
medium_value_spend_threshold = (
    returning_value_base[
        'Total Spend To Date'
    ].quantile(0.75)
)

high_value_spend_threshold = (
    returning_value_base[
        'Total Spend To Date'
    ].quantile(0.90)
)

high_ticket_aov_threshold = (
    new_customer_value_base[
        'Customer AOV To Date'
    ].quantile(0.90)
)

print(
    "Medium Value Spend Threshold:",
    medium_value_spend_threshold
)

print(
    "High Value Spend Threshold:",
    high_value_spend_threshold
)

print(
    "High Ticket AOV Threshold:",
    high_ticket_aov_threshold
)


In [ ]:
def customer_value_segment(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    total_orders = row['Total Orders To Date']
    total_spend = row['Total Spend To Date']
    customer_aov = row['Customer AOV To Date']

    if total_orders == 1:
        if customer_aov >= high_ticket_aov_threshold:
            return 'عميل جديد - طلب أول مرتفع'

        return 'عميل جديد - طلب أول عادي'

    if total_spend >= high_value_spend_threshold:
        return 'عميل متكرر عالي الإنفاق'

    if total_spend >= medium_value_spend_threshold:
        return 'عميل متكرر متوسط الإنفاق'

    return 'عميل متكرر منخفض الإنفاق'


final_marketing['Customer Value Segment'] = (
    final_marketing.apply(
        customer_value_segment,
        axis=1
    )
)


In [ ]:
display(
    final_marketing[
        'Customer Value Segment'
    ].value_counts()
)


In [ ]:
# =========================
# 4) Customer frequency segment
# =========================

def customer_frequency_segment(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    orders = row['Total Orders To Date']

    if orders >= 8:
        return 'عميل متكرر جدًا - 8 طلبات فأكثر'

    if orders >= 4:
        return 'عميل متكرر - من 4 إلى 7 طلبات'

    if orders >= 2:
        return 'عميل محدود التكرار - طلبان أو 3 طلبات'

    return 'عميل بطلب واحد'


final_marketing['Customer Frequency Segment'] = (
    final_marketing.apply(
        customer_frequency_segment,
        axis=1
    )
)


In [ ]:
display(
    final_marketing['Customer Frequency Segment'].value_counts()
)



In [ ]:
# =========================
# 5) Inactivity segment
# =========================

# آخر تاريخ موجود في بيانات الطلبات
analysis_reference_date = (
    pd.to_datetime(
        order_features['order_date'],
        errors='coerce'
    )
    .max()
    .normalize()
)

# تاريخ آخر طلب لكل عميل
final_marketing['Last Order Date'] = pd.to_datetime(
    final_marketing['order_date'],
    errors='coerce'
)

# عدد الأيام من آخر طلب للعميل حتى نهاية البيانات
final_marketing['Days Since Last Order'] = (
    analysis_reference_date
    - final_marketing['Last Order Date']
).dt.days


def inactivity_segment(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    days = row['Days Since Last Order']

    if pd.isna(days):
        return 'تاريخ آخر طلب غير متوفر'

    if days <= 14:
        return 'نشط مؤخرًا - آخر طلب خلال 14 يومًا'

    if days <= 30:
        return 'نشط بشكل طبيعي - آخر طلب منذ 15 إلى 30 يومًا'

    if days <= 60:
        return 'بدأ يتأخر - آخر طلب منذ 31 إلى 60 يومًا'

    if days <= 90:
        return 'عميل خامل - آخر طلب منذ 61 إلى 90 يومًا'

    return 'عميل خامل لفترة طويلة - آخر طلب منذ أكثر من 90 يومًا'


final_marketing['Inactivity Segment'] = (
    final_marketing.apply(
        inactivity_segment,
        axis=1
    )
)



In [ ]:
display(
    final_marketing[
        'Inactivity Segment'
    ].value_counts()
)


In [ ]:
# =========================
# 6) Financial behavior by discount type
# =========================

# نأخذ جميع طلبات العملاء مع أنواع الخصومات المنفصلة
financial_behavior_base = order_features[[
    'customer_id',
    'order_id',
    'financial_amount_promocode_discount',
    'financial_amount_gift_wallet_discount',
    'financial_amount_loyalty_discount',
    'financial_amount_subscription_discount',
    'financial_amount_refund'
]].copy()


# -------------------------------------------------
# 6.1) تنظيف أعمدة المبالغ
# -------------------------------------------------

financial_amount_columns = [
    'financial_amount_promocode_discount',
    'financial_amount_gift_wallet_discount',
    'financial_amount_loyalty_discount',
    'financial_amount_subscription_discount',
    'financial_amount_refund'
]

for column in financial_amount_columns:

    financial_behavior_base[column] = (
        pd.to_numeric(
            financial_behavior_base[column],
            errors='coerce'
        )
        .fillna(0)
        .clip(lower=0)
    )


# -------------------------------------------------
# 6.2) تحديد ما الذي استُخدم في كل طلب
# -------------------------------------------------

financial_behavior_base['used_promocode'] = (
    financial_behavior_base[
        'financial_amount_promocode_discount'
    ] > 0
).astype(int)

financial_behavior_base['used_gift_wallet'] = (
    financial_behavior_base[
        'financial_amount_gift_wallet_discount'
    ] > 0
).astype(int)

financial_behavior_base['used_loyalty'] = (
    financial_behavior_base[
        'financial_amount_loyalty_discount'
    ] > 0
).astype(int)

financial_behavior_base['used_subscription_benefit'] = (
    financial_behavior_base[
        'financial_amount_subscription_discount'
    ] > 0
).astype(int)

financial_behavior_base['received_refund'] = (
    financial_behavior_base[
        'financial_amount_refund'
    ] > 0
).astype(int)


# -------------------------------------------------
# 6.3) تجهيز الحوافز التسويقية
# -------------------------------------------------

# الحوافز التسويقية:
# Promocode Discount + Gift Wallet

financial_behavior_base['marketing_incentive_amount'] = (
    financial_behavior_base[
        'financial_amount_promocode_discount'
    ]
    +
    financial_behavior_base[
        'financial_amount_gift_wallet_discount'
    ]
)

# لو الطلب استخدم البروموكود والمحفظة معًا،
# نحسبه طلب حوافز واحد فقط
financial_behavior_base['used_marketing_incentive'] = (
    (
        financial_behavior_base['used_promocode'] > 0
    )
    |
    (
        financial_behavior_base['used_gift_wallet'] > 0
    )
).astype(int)


# -------------------------------------------------
# 6.4) تجميع التاريخ المالي لكل عميل
# -------------------------------------------------

customer_financial_behavior = (
    financial_behavior_base
    .groupby('customer_id')
    .agg(
        financial_analysis_orders=(
            'order_id',
            'nunique'
        ),

        marketing_incentive_orders=(
            'used_marketing_incentive',
            'sum'
        ),

        marketing_incentive_amount=(
            'marketing_incentive_amount',
            'sum'
        ),

        promocode_orders=(
            'used_promocode',
            'sum'
        ),

        promocode_amount=(
            'financial_amount_promocode_discount',
            'sum'
        ),

        gift_wallet_orders=(
            'used_gift_wallet',
            'sum'
        ),

        gift_wallet_amount=(
            'financial_amount_gift_wallet_discount',
            'sum'
        ),

        loyalty_usage_orders=(
            'used_loyalty',
            'sum'
        ),

        loyalty_discount_amount=(
            'financial_amount_loyalty_discount',
            'sum'
        ),

        subscription_benefit_orders=(
            'used_subscription_benefit',
            'sum'
        ),

        subscription_discount_amount=(
            'financial_amount_subscription_discount',
            'sum'
        ),

        refund_orders=(
            'received_refund',
            'sum'
        ),

        refund_amount=(
            'financial_amount_refund',
            'sum'
        )
    )
    .reset_index()
)


# -------------------------------------------------
# 6.5) دمج النتائج مع جدول التسويق
# -------------------------------------------------

financial_behavior_columns = [
    column
    for column in customer_financial_behavior.columns
    if column != 'customer_id'
]

# حذف الأعمدة القديمة لو أعدنا تشغيل البلوك
existing_financial_behavior_columns = [
    column
    for column in financial_behavior_columns
    if column in final_marketing.columns
]

if existing_financial_behavior_columns:

    final_marketing = final_marketing.drop(
        columns=existing_financial_behavior_columns
    )


final_marketing = final_marketing.merge(
    customer_financial_behavior,
    on='customer_id',
    how='left'
)

final_marketing[
    financial_behavior_columns
] = (
    final_marketing[
        financial_behavior_columns
    ]
    .fillna(0)
)


# -------------------------------------------------
# 6.6) نسبة الطلبات التي استُخدمت فيها حوافز تسويقية
# -------------------------------------------------

final_marketing['Marketing Incentive Order Ratio'] = (
    np.where(
        final_marketing[
            'financial_analysis_orders'
        ] > 0,

        final_marketing[
            'marketing_incentive_orders'
        ]
        /
        final_marketing[
            'financial_analysis_orders'
        ],

        0
    )
)


# -------------------------------------------------
# 6.7) تصنيف استخدام الحوافز التسويقية
# -------------------------------------------------

def marketing_incentive_usage_segment(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    total_orders = row[
        'financial_analysis_orders'
    ]

    incentive_orders = row[
        'marketing_incentive_orders'
    ]

    incentive_ratio = row[
        'Marketing Incentive Order Ratio'
    ]

    if total_orders == 1:

        if incentive_orders > 0:
            return 'استفاد من عرض تسويقي في طلبه الوحيد'

        return 'لم يستفد من عرض تسويقي في طلبه الوحيد'

    if incentive_ratio >= 0.50:
        return 'استفادة مرتفعة من العروض التسويقية - في نصف طلباته فأكثر'

    if incentive_ratio >= 0.25:
        return 'استفادة متوسطة من العروض التسويقية - من ربع طلباته إلى أقل من النصف'

    if incentive_ratio > 0:
        return 'استفادة محدودة من العروض التسويقية - في أقل من ربع طلباته'

    return 'لم يستفد من عروض تسويقية'


final_marketing['Marketing Incentive Usage Segment'] = (
    final_marketing.apply(
        marketing_incentive_usage_segment,
        axis=1
    )
)


# -------------------------------------------------
# 6.8) حالة استخدام نقاط الولاء
# -------------------------------------------------

def loyalty_usage_status(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    if row['loyalty_usage_orders'] > 0:
        return 'استخدم نقاط الولاء سابقًا'

    return 'لم يستخدم نقاط الولاء'


final_marketing['Loyalty Usage Status'] = (
    final_marketing.apply(
        loyalty_usage_status,
        axis=1
    )
)


# -------------------------------------------------
# 6.9) حالة الاستفادة من خصم الاشتراك
# -------------------------------------------------

def subscription_benefit_status(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    active_subscription = row.get(
        'has_active_subscription',
        False
    )

    used_subscription_benefit = (
        row['subscription_benefit_orders'] > 0
    )

    if active_subscription and used_subscription_benefit:
        return 'مشترك حاليًا واستفاد من خصم الاشتراك'

    if active_subscription:
        return 'مشترك حاليًا - لا توجد استفادة مسجلة من الخصم'

    if used_subscription_benefit:
        return 'استفاد من خصم الاشتراك سابقًا'

    return 'لم يستفد من خصم الاشتراك'


final_marketing['Subscription Benefit Status'] = (
    final_marketing.apply(
        subscription_benefit_status,
        axis=1
    )
)


# -------------------------------------------------
# 6.10) حالة التعويض أو الاسترجاع
# -------------------------------------------------

def refund_history_status(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    if row['refund_orders'] > 0:
        return 'لديه تعويض أو استرجاع مسجل سابقًا'

    return 'لا يوجد تعويض أو استرجاع مسجل'


final_marketing['Refund History Status'] = (
    final_marketing.apply(
        refund_history_status,
        axis=1
    )
)


In [ ]:
# -------------------------------------------------
# 6.12) فحوصات النتيجة
# -------------------------------------------------

print("Marketing incentive usage:")
display(
    final_marketing[
        'Marketing Incentive Usage Segment'
    ].value_counts()
)

print("\nLoyalty usage:")
display(
    final_marketing[
        'Loyalty Usage Status'
    ].value_counts()
)

print("\nSubscription benefits:")
display(
    final_marketing[
        'Subscription Benefit Status'
    ].value_counts()
)

print("\nRefund history:")
display(
    final_marketing[
        'Refund History Status'
    ].value_counts()
)


## 11. Subscription / Package Recommendation Logic

Map customer behavior to a portfolio-safe package recommendation framework. Public labels are anonymized as **Package A / B / C**.


In [ ]:
# =========================
# 7.0) Package catalog
# =========================

package_catalog = {

    'package_a': {
        'package_name': 'الباقة الشبابية',
        'package_price': 199,

        'unlimited_laundry': True,
        'monthly_visits': 4,
        'extra_monthly_visits': 0,

        'monthly_blankets': 1,
        'monthly_shoes': 2,

        'pickup_delivery_type': 'زيارة الفرع',
        'free_home_pickup_delivery': False,

        'service_bags_count': 4,
        'service_bag_type': 'Service Bags',

        'monthly_car_washes': 0,

        'daily_express_ironing': True,
        'daily_express_shemagh': 1,
        'daily_express_thobe': 1
    },


    'package_b': {
        'package_name': 'الباقة الشبابية بلس',
        'package_price': 299,

        'unlimited_laundry': True,
        'monthly_visits': 4,
        'extra_monthly_visits': 0,

        'monthly_blankets': 2,
        'monthly_shoes': 3,

        'pickup_delivery_type': 'استلام وتسليم من المنزل',
        'free_home_pickup_delivery': True,

        'service_bags_count': 1,
        'service_bag_type': 'Service Bag',

        'monthly_car_washes': 1,

        'daily_express_ironing': False,
        'daily_express_shemagh': 0,
        'daily_express_thobe': 0
    },


    'package_c': {
        'package_name': 'الباقة العائلية',
        'package_price': 399,

        'unlimited_laundry': True,
        'monthly_visits': 4,
        'extra_monthly_visits': 1,

        'monthly_blankets': 5,
        'monthly_shoes': 5,

        'pickup_delivery_type': 'استلام وتسليم من المنزل',
        'free_home_pickup_delivery': True,

        'service_bags_count': 2,
        'service_bag_type': 'Service Bag',

        'monthly_car_washes': 2,

        'daily_express_ironing': False,
        'daily_express_shemagh': 0,
        'daily_express_thobe': 0
    }
}


In [ ]:
# =========================
# 7.1) Package purchase history by customer
# =========================

customer_package_base = order_features[[
    'customer_id',
    'order_id',
    'order_date',
    'package',
    'financial_amount_package'
]].copy()


# -------------------------------------------------
# تنظيف الأعمدة
# -------------------------------------------------

customer_package_base['order_date'] = pd.to_datetime(
    customer_package_base['order_date'],
    errors='coerce'
)

customer_package_base['package'] = (
    pd.to_numeric(
        customer_package_base['package'],
        errors='coerce'
    )
    .fillna(0)
)

customer_package_base['financial_amount_package'] = (
    pd.to_numeric(
        customer_package_base['financial_amount_package'],
        errors='coerce'
    )
    .fillna(0)
    .clip(lower=0)
)


# -------------------------------------------------
# تحديد هل الطلب يحتوي شراء باقة
# -------------------------------------------------

customer_package_base['used_package'] = (
    (customer_package_base['package'] > 0)
    |
    (
        customer_package_base[
            'financial_amount_package'
        ] > 0
    )
).astype(int)


# -------------------------------------------------
# تجميع تاريخ الباقات لكل عميل
# -------------------------------------------------

customer_package_history = (
    customer_package_base
    .groupby('customer_id')
    .agg(
        package_purchase_orders=(
            'used_package',
            'sum'
        ),

        package_purchase_amount=(
            'financial_amount_package',
            'sum'
        )
    )
    .reset_index()
)


# -------------------------------------------------
# إنشاء حالة بسيطة: هل استخدم باقة سابقًا؟
# -------------------------------------------------

customer_package_history['Has Used Package Before'] = (
    customer_package_history[
        'package_purchase_orders'
    ] > 0
)


In [ ]:
# =========================
# 7.2) Merge package history with final marketing table
# =========================

package_history_columns = [
    'package_purchase_orders',
    'package_purchase_amount',
    'Has Used Package Before'
]


# -------------------------------------------------
# حذف الأعمدة القديمة لو أعدنا تشغيل البلوك
# -------------------------------------------------

existing_package_history_columns = [
    column
    for column in package_history_columns
    if column in final_marketing.columns
]

if existing_package_history_columns:

    final_marketing = final_marketing.drop(
        columns=existing_package_history_columns
    )


# -------------------------------------------------
# دمج تاريخ الباقات حسب customer_id
# -------------------------------------------------

final_marketing = final_marketing.merge(
    customer_package_history,
    on='customer_id',
    how='left'
)


# -------------------------------------------------
# تعبئة وتنظيف القيم بعد الدمج
# -------------------------------------------------

final_marketing['package_purchase_orders'] = (
    final_marketing['package_purchase_orders']
    .fillna(0)
    .astype(int)
)

final_marketing['package_purchase_amount'] = (
    final_marketing['package_purchase_amount']
    .fillna(0)
)

final_marketing['Has Used Package Before'] = (
    final_marketing['Has Used Package Before']
    .fillna(False)
    .astype(bool)
)


In [ ]:
display(
    final_marketing[
        'Has Used Package Before'
    ].value_counts()
)

display(
    final_marketing[[
        'has_active_subscription',
        'Has Used Package Before'
    ]]
    .value_counts()
)


In [ ]:
# =========================
# 7.3) Classify products for package analysis
# =========================

# -------------------------------------------------
# 1) القطع التي تدخل ضمن الغسيل اللامحدود
# -------------------------------------------------

unlimited_laundry_columns = [
    'qty_ordered_thoab',
    'qty_ordered_boxer',
    'qty_ordered_shirt',
    'qty_ordered_undershirt',
    'qty_ordered_t_shirt',
    'qty_ordered_shummagh_gutrah',
    'qty_ordered_trouser',
    'qty_ordered_socks',
    'qty_ordered_serwal',
    'qty_ordered_pillow_case',
    'qty_ordered_thobe_winter',
    'qty_ordered_short',
    'qty_ordered_color_thoab',
    'qty_ordered_underwear',
    'qty_ordered_niqab_hijab',
    'qty_ordered_bed_sheet_small',
    'qty_ordered_taqia',
    'qty_ordered_bed_sheet_big',
    'qty_ordered_towel',
    'qty_ordered_hoodie',
    'qty_ordered_hand_towel',
    'qty_ordered_dress',
    'qty_ordered_bathrobe',
    'qty_ordered_pullover',
    'qty_ordered_blouse',
    'qty_ordered_ladies_shirt',
    'qty_ordered_bed_cover_king',
    'qty_ordered_towel_medium',
    'qty_ordered_scarf',
    'qty_ordered_thobe_small',
    'qty_ordered_lab_coat',
    'qty_ordered_kids_shirt',
    'qty_ordered_bed_cover_single',
    'qty_ordered_bejama',
    'qty_ordered_bed_cover_queen',
    'qty_ordered_face_towel',
    'qty_ordered_maryol',
    'qty_ordered_vest',
    'qty_ordered_kids_trouser',
    'qty_ordered_skirt',
    'qty_ordered_bra',
    'qty_ordered_overall',
    'qty_ordered_cap',
    'qty_ordered_uniforms',
    'qty_ordered_ihram',
    'qty_ordered_seat_cover_small',
    'qty_ordered_winter_ghotrah',
    'qty_ordered_ezar',
    'qty_ordered_thobe_winter_small',
    'qty_ordered_kids_dress',
    'qty_ordered_shirt_sport',
    'qty_ordered_hijab',
    'qty_ordered_tie',
    'qty_ordered_seat_cover_big',
    'qty_ordered_short_sport',
    'qty_ordered_duvet_cover',
    'qty_ordered_infant_bodysuit',
    'qty_ordered_trouser_sport',
    'qty_ordered_ladies_trouser',
    'qty_ordered_table_cover',
    'qty_ordered_socks_sport',
    'qty_ordered_night_dress',
    'qty_ordered_sleep_dress',
    'qty_ordered_mask',
    'qty_ordered_gloves_2_pc',
    'qty_ordered_bed_spred_small',
    'qty_ordered_track_suits'
]


# -------------------------------------------------
# 2) البطانيات واللحاف
# ميزة مستقلة داخل الباقات
# -------------------------------------------------

blanket_benefit_columns = [
    'qty_ordered_blanket_small',
    'qty_ordered_blanket_big',
    'qty_ordered_medium_blanket',
    'qty_ordered_small_blanket',
    'qty_ordered_duvet'
]


# -------------------------------------------------
# 3) الأحذية
# ميزة مستقلة داخل الباقات
# -------------------------------------------------

shoe_benefit_columns = [
    'qty_ordered_shoes',
    'qty_ordered_shoes_sport',
    'qty_ordered_shoe'
]


# -------------------------------------------------
# 4) قطع وخدمات لا تدخل في الغسيل اللامحدود
# ولا تُحسب ضمن مزايا البطانيات أو الأحذية
# -------------------------------------------------

excluded_package_product_columns = [
    'qty_ordered_carpet',
    'qty_ordered_abaya',
    'qty_ordered_jacket',
    'qty_ordered_restaurant_items',
    'qty_ordered_military_suit',
    'qty_ordered_pakistani_suit',
    'qty_ordered_pillow',
    'qty_ordered_men_suit_2_pes',
    'qty_ordered_meshlah',
    'qty_ordered_curtain',
    'qty_ordered_bath_mate',
    'qty_ordered_overcoat',
    'qty_ordered_prayer_rug',
    'qty_ordered_felt',
    'qty_ordered_fur',
    'qty_ordered_spot_service',
    'qty_ordered_crystal_dress',
    'qty_ordered_washing_service',
    'qty_ordered_coat',
    'qty_ordered_mattress_medium',
    'qty_ordered_wool',
    'qty_ordered_mattress_big',
    'qty_ordered_women_suit',
    'qty_ordered_wedding_dress',
    'qty_ordered_mattress_small',
    'qty_ordered_suit_3_pcs',
    'qty_ordered_abaya_scarf',
    'qty_ordered_special_carpets',
    'qty_ordered_mop',
    'qty_ordered_military_jacket'
]


# -------------------------------------------------
# 5) فحص أن كل أعمدة القطع مصنفة مرة واحدة فقط
# -------------------------------------------------

all_product_quantity_columns = [
    column
    for column in order_features.columns
    if column.startswith('qty_ordered_')
]

classified_product_columns = (
    unlimited_laundry_columns
    + blanket_benefit_columns
    + shoe_benefit_columns
    + excluded_package_product_columns
)

missing_product_columns = sorted(
    set(all_product_quantity_columns)
    -
    set(classified_product_columns)
)

duplicated_product_columns = sorted({
    column
    for column in classified_product_columns
    if classified_product_columns.count(column) > 1
})

unknown_classified_columns = sorted(
    set(classified_product_columns)
    -
    set(all_product_quantity_columns)
)


print(
    "Unlimited laundry columns:",
    len(unlimited_laundry_columns)
)

print(
    "Blanket benefit columns:",
    len(blanket_benefit_columns)
)

print(
    "Shoe benefit columns:",
    len(shoe_benefit_columns)
)

print(
    "Excluded product columns:",
    len(excluded_package_product_columns)
)

print(
    "Total product columns in data:",
    len(all_product_quantity_columns)
)

print(
    "Missing product columns:",
    missing_product_columns
)

print(
    "Duplicated product columns:",
    duplicated_product_columns
)

print(
    "Unknown classified columns:",
    unknown_classified_columns
)


In [ ]:
# =========================
# 7.4) Package usage features per order
# =========================

package_usage_base = order_features[[
    'customer_id',
    'order_id',
    'order_date',
    'order_category'
] + classified_product_columns].copy()


# -------------------------------------------------
# 1) تنظيف تاريخ الطلب
# -------------------------------------------------

package_usage_base['order_date'] = pd.to_datetime(
    package_usage_base['order_date'],
    errors='coerce'
)


# -------------------------------------------------
# 2) تنظيف جميع أعمدة كميات القطع دفعة واحدة
# -------------------------------------------------

package_usage_base[classified_product_columns] = (
    package_usage_base[
        classified_product_columns
    ]
    .apply(
        pd.to_numeric,
        errors='coerce'
    )
    .fillna(0)
    .clip(lower=0)
)


# إعادة تنظيم الجدول داخليًا لتجنب PerformanceWarning
package_usage_base = package_usage_base.copy()


# -------------------------------------------------
# 3) إنشاء خصائص الطلب دفعة واحدة
# -------------------------------------------------

package_usage_features = pd.DataFrame(
    index=package_usage_base.index
)

package_usage_features['unlimited_laundry_quantity'] = (
    package_usage_base[
        unlimited_laundry_columns
    ]
    .sum(axis=1)
)

package_usage_features['blanket_benefit_quantity'] = (
    package_usage_base[
        blanket_benefit_columns
    ]
    .sum(axis=1)
)

package_usage_features['shoe_benefit_quantity'] = (
    package_usage_base[
        shoe_benefit_columns
    ]
    .sum(axis=1)
)

package_usage_features['excluded_package_quantity'] = (
    package_usage_base[
        excluded_package_product_columns
    ]
    .sum(axis=1)
)


# -------------------------------------------------
# 4) هل ظهر كل نوع داخل الطلب؟
# -------------------------------------------------

package_usage_features['has_unlimited_laundry_items'] = (
    package_usage_features[
        'unlimited_laundry_quantity'
    ] > 0
).astype(int)

package_usage_features['has_blanket_benefit_items'] = (
    package_usage_features[
        'blanket_benefit_quantity'
    ] > 0
).astype(int)

package_usage_features['has_shoe_benefit_items'] = (
    package_usage_features[
        'shoe_benefit_quantity'
    ] > 0
).astype(int)


# -------------------------------------------------
# 5) هل الطلب أونلاين؟
# -------------------------------------------------

package_usage_features['is_online_order'] = (
    package_usage_base[
        'order_category'
    ]
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('online')
).astype(int)


# -------------------------------------------------
# 6) إضافة الخصائص إلى الجدول دفعة واحدة
# -------------------------------------------------

package_usage_base = pd.concat(
    [
        package_usage_base,
        package_usage_features
    ],
    axis=1
)


# -------------------------------------------------
# 7) فحص سريع
# -------------------------------------------------

print(
    "Orders with unlimited laundry items:",
    int(
        package_usage_base[
            'has_unlimited_laundry_items'
        ].sum()
    )
)

print(
    "Orders with blankets or duvet:",
    int(
        package_usage_base[
            'has_blanket_benefit_items'
        ].sum()
    )
)

print(
    "Orders with shoes:",
    int(
        package_usage_base[
            'has_shoe_benefit_items'
        ].sum()
    )
)

display(
    package_usage_base[[
        'customer_id',
        'order_id',
        'order_date',
        'order_category',
        'unlimited_laundry_quantity',
        'blanket_benefit_quantity',
        'shoe_benefit_quantity',
        'excluded_package_quantity',
        'is_online_order'
    ]].head(10)
)


In [ ]:
# =========================
# 7.5) Package usage history by customer
# =========================

# نأخذ فقط الأعمدة التي نحتاجها من جدول الطلبات
package_customer_usage_base = package_usage_base[[
    'customer_id',
    'order_id',
    'order_date',
    'unlimited_laundry_quantity',
    'blanket_benefit_quantity',
    'shoe_benefit_quantity',
    'excluded_package_quantity',
    'is_online_order'
]].copy()


# -------------------------------------------------
# 1) توحيد تاريخ الطلب بدون الساعة
# -------------------------------------------------

package_customer_usage_base['order_day'] = (
    pd.to_datetime(
        package_customer_usage_base['order_date'],
        errors='coerce'
    )
    .dt.normalize()
)


# -------------------------------------------------
# 2) تحديد آخر 90 يومًا
# -------------------------------------------------

recent_package_start_date = (
    analysis_reference_date
    - pd.Timedelta(days=89)
)

recent_package_mask = (
    (
        package_customer_usage_base['order_day']
        >= recent_package_start_date
    )
    &
    (
        package_customer_usage_base['order_day']
        <= analysis_reference_date
    )
)


# -------------------------------------------------
# 3) التاريخ الكامل لكل عميل
# -------------------------------------------------

customer_full_package_usage = (
    package_customer_usage_base
    .groupby('customer_id')
    .agg(
        package_usage_analysis_orders=(
            'order_id',
            'nunique'
        ),

        total_unlimited_laundry_quantity=(
            'unlimited_laundry_quantity',
            'sum'
        ),

        total_blanket_benefit_quantity=(
            'blanket_benefit_quantity',
            'sum'
        ),

        total_shoe_benefit_quantity=(
            'shoe_benefit_quantity',
            'sum'
        ),

        total_excluded_package_quantity=(
            'excluded_package_quantity',
            'sum'
        ),

        total_online_orders=(
            'is_online_order',
            'sum'
        )
    )
    .reset_index()
)


# -------------------------------------------------
# 4) استخدام آخر 90 يومًا لكل عميل
# -------------------------------------------------

customer_recent_package_usage = (
    package_customer_usage_base.loc[
        recent_package_mask
    ]
    .groupby('customer_id')
    .agg(
        recent_90d_orders=(
            'order_id',
            'nunique'
        ),

        recent_90d_unlimited_laundry_quantity=(
            'unlimited_laundry_quantity',
            'sum'
        ),

        recent_90d_blanket_benefit_quantity=(
            'blanket_benefit_quantity',
            'sum'
        ),

        recent_90d_shoe_benefit_quantity=(
            'shoe_benefit_quantity',
            'sum'
        ),

        recent_90d_excluded_package_quantity=(
            'excluded_package_quantity',
            'sum'
        ),

        recent_90d_online_orders=(
            'is_online_order',
            'sum'
        )
    )
    .reset_index()
)


# -------------------------------------------------
# 5) دمج التاريخ الكامل مع آخر 90 يومًا
# -------------------------------------------------

customer_package_usage = (
    customer_full_package_usage
    .merge(
        customer_recent_package_usage,
        on='customer_id',
        how='left'
    )
)


package_usage_numeric_columns = [
    column
    for column in customer_package_usage.columns
    if column != 'customer_id'
]

customer_package_usage[
    package_usage_numeric_columns
] = (
    customer_package_usage[
        package_usage_numeric_columns
    ]
    .fillna(0)
)


# -------------------------------------------------
# 6) حساب المتوسطات الشهرية الحديثة
# آخر 90 يومًا = 3 أشهر تقريبًا
# -------------------------------------------------

customer_package_usage[
    'Recent Monthly Order Average'
] = (
    customer_package_usage[
        'recent_90d_orders'
    ] / 3
)

customer_package_usage[
    'Recent Monthly Unlimited Laundry Average'
] = (
    customer_package_usage[
        'recent_90d_unlimited_laundry_quantity'
    ] / 3
)

customer_package_usage[
    'Recent Monthly Blanket Average'
] = (
    customer_package_usage[
        'recent_90d_blanket_benefit_quantity'
    ] / 3
)

customer_package_usage[
    'Recent Monthly Shoe Average'
] = (
    customer_package_usage[
        'recent_90d_shoe_benefit_quantity'
    ] / 3
)


# -------------------------------------------------
# 7) دمج النتائج مع final_marketing
# -------------------------------------------------

customer_package_usage_columns = [
    column
    for column in customer_package_usage.columns
    if column != 'customer_id'
]


existing_customer_package_usage_columns = [
    column
    for column in customer_package_usage_columns
    if column in final_marketing.columns
]

if existing_customer_package_usage_columns:

    final_marketing = final_marketing.drop(
        columns=existing_customer_package_usage_columns
    )


final_marketing = final_marketing.merge(
    customer_package_usage,
    on='customer_id',
    how='left'
)


final_marketing[
    customer_package_usage_columns
] = (
    final_marketing[
        customer_package_usage_columns
    ]
    .fillna(0)
)


# -------------------------------------------------
# 8) فحص النتيجة
# -------------------------------------------------

print(
    "Recent package usage period:",
    recent_package_start_date.date(),
    "to",
    analysis_reference_date.date()
)

print(
    "Customers in package usage table:",
    len(customer_package_usage)
)

print(
    "Rows in final_marketing:",
    len(final_marketing)
)

display(
    final_marketing[[
        'customer_id',

        'package_usage_analysis_orders',
        'total_unlimited_laundry_quantity',
        'total_blanket_benefit_quantity',
        'total_shoe_benefit_quantity',

        'recent_90d_orders',
        'recent_90d_unlimited_laundry_quantity',
        'recent_90d_blanket_benefit_quantity',
        'recent_90d_shoe_benefit_quantity',

        'Recent Monthly Order Average',
        'Recent Monthly Unlimited Laundry Average',
        'Recent Monthly Blanket Average',
        'Recent Monthly Shoe Average'
    ]].head(10)
)


In [ ]:
# =========================
# 7.6) Package audience status
# =========================

def package_audience_status(row):

    if row['Suspected Non-Individual Account']:
        return 'حساب بطلبات يومية غير طبيعية'

    if row['has_active_subscription']:
        return 'مشترك حاليًا - لا يعرض عليه اشتراك جديد'

    if row['Has Used Package Before']:
        return 'استخدم باقة سابقًا - جمهور التجديد'

    return 'لم يستخدم باقة سابقًا - جمهور أول اشتراك'


final_marketing['Package Audience Status'] = (
    final_marketing.apply(
        package_audience_status,
        axis=1
    )
)


display(
    final_marketing[
        'Package Audience Status'
    ].value_counts()
)


In [ ]:
# =========================
# 7.7) Identify previous package type
# =========================

# نأخذ طلبات العملاء مع معلومات شراء الباقات
package_purchase_detail = order_features[[
    'customer_id',
    'order_id',
    'order_date',
    'package',
    'financial_amount_package'
]].copy()


# -------------------------------------------------
# 1) تنظيف الأعمدة
# -------------------------------------------------

package_purchase_detail['order_date'] = pd.to_datetime(
    package_purchase_detail['order_date'],
    errors='coerce'
)

package_purchase_detail['package'] = (
    pd.to_numeric(
        package_purchase_detail['package'],
        errors='coerce'
    )
    .fillna(0)
)

package_purchase_detail['financial_amount_package'] = (
    pd.to_numeric(
        package_purchase_detail[
            'financial_amount_package'
        ],
        errors='coerce'
    )
    .fillna(0)
    .clip(lower=0)
)


# -------------------------------------------------
# 2) الاحتفاظ بطلبات شراء الباقات فقط
# -------------------------------------------------

package_purchase_detail = package_purchase_detail[
    (
        package_purchase_detail['package'] > 0
    )
    |
    (
        package_purchase_detail[
            'financial_amount_package'
        ] > 0
    )
].copy()


# -------------------------------------------------
# 3) تحديد نوع الباقة من مبلغ الشراء
# -------------------------------------------------

def identify_package_from_amount(amount):

    # Package A
    # 199 بدون الضريبة
    # 228.85 مع الضريبة
    # 128.85 مع الضريبة وبعد خصم 100 ريال
    youth_package_amounts = [
        128.85,
        199.00,
        228.85
    ]

    # Package B
    # 299 بدون الضريبة
    # 343.85 مع الضريبة
    # 243.85 مع الضريبة وبعد خصم 100 ريال
    youth_plus_package_amounts = [
        243.85,
        299.00,
        343.85
    ]

    # Package C
    # 399 بدون الضريبة
    # 458.85 مع الضريبة
    # 358.85 مع الضريبة وبعد خصم 100 ريال
    family_package_amounts = [
        358.85,
        399.00,
        458.85
    ]

    if any(
        np.isclose(
            amount,
            package_amount,
            atol=0.01
        )
        for package_amount in youth_package_amounts
    ):
        return 'Package A'

    if any(
        np.isclose(
            amount,
            package_amount,
            atol=0.01
        )
        for package_amount in youth_plus_package_amounts
    ):
        return 'Package B'

    if any(
        np.isclose(
            amount,
            package_amount,
            atol=0.01
        )
        for package_amount in family_package_amounts
    ):
        return 'Package C'

    return 'باقة غير محددة من المبلغ'


package_purchase_detail['Previous Package Type'] = (
    package_purchase_detail[
        'financial_amount_package'
    ]
    .apply(
        identify_package_from_amount
    )
)


# -------------------------------------------------
# 4) فصل عمليات الشراء المعروفة وغير المعروفة
# -------------------------------------------------

known_package_purchases = package_purchase_detail[
    package_purchase_detail[
        'Previous Package Type'
    ] != 'باقة غير محددة من المبلغ'
].copy()


unknown_package_purchases = package_purchase_detail[
    package_purchase_detail[
        'Previous Package Type'
    ] == 'باقة غير محددة من المبلغ'
].copy()


# -------------------------------------------------
# 5) استخراج آخر باقة معروفة لكل عميل
# -------------------------------------------------

latest_known_package = (
    known_package_purchases
    .sort_values(
        [
            'customer_id',
            'order_date',
            'order_id'
        ],
        na_position='first'
    )
    .groupby(
        'customer_id',
        as_index=False
    )
    .tail(1)
    [[
        'customer_id',
        'order_date',
        'financial_amount_package',
        'Previous Package Type'
    ]]
    .rename(
        columns={
            'order_date':
                'Last Known Package Purchase Date',

            'financial_amount_package':
                'Last Known Package Purchase Amount',

            'Previous Package Type':
                'Last Known Package'
        }
    )
)


# -------------------------------------------------
# 6) حساب عدد عمليات شراء الباقات المعروفة لكل عميل
# -------------------------------------------------

known_package_counts = (
    known_package_purchases
    .groupby('customer_id')
    .size()
    .rename(
        'Known Package Purchase Orders'
    )
    .reset_index()
)


# -------------------------------------------------
# 7) حساب عدد عمليات الشراء غير محددة النوع
# -------------------------------------------------

unknown_package_counts = (
    unknown_package_purchases
    .groupby('customer_id')
    .size()
    .rename(
        'Unknown Package Purchase Orders'
    )
    .reset_index()
)


# -------------------------------------------------
# 8) إنشاء جدول تاريخ الباقة السابقة للعملاء
# -------------------------------------------------

customer_previous_package = (
    customer_package_history[[
        'customer_id'
    ]]
    .merge(
        latest_known_package,
        on='customer_id',
        how='left'
    )
    .merge(
        known_package_counts,
        on='customer_id',
        how='left'
    )
    .merge(
        unknown_package_counts,
        on='customer_id',
        how='left'
    )
)


customer_previous_package[
    [
        'Known Package Purchase Orders',
        'Unknown Package Purchase Orders'
    ]
] = (
    customer_previous_package[
        [
            'Known Package Purchase Orders',
            'Unknown Package Purchase Orders'
        ]
    ]
    .fillna(0)
    .astype(int)
)


# -------------------------------------------------
# 9) حذف الأعمدة القديمة لو أعدنا تشغيل البلوك
# -------------------------------------------------

previous_package_columns_to_remove = [
    'Last Known Package Purchase Date',
    'Last Known Package Purchase Amount',
    'Last Known Package',
    'Known Package Purchase Orders',
    'Unknown Package Purchase Orders',

    # أسماء قديمة احتياطًا
    'Last Package Purchase Date',
    'Last Package Purchase Amount'
]


existing_previous_package_columns = [
    column
    for column in previous_package_columns_to_remove
    if column in final_marketing.columns
]


if existing_previous_package_columns:

    final_marketing = final_marketing.drop(
        columns=existing_previous_package_columns
    )


# -------------------------------------------------
# 10) دمج تاريخ الباقة السابقة مع final_marketing
# -------------------------------------------------

final_marketing = final_marketing.merge(
    customer_previous_package,
    on='customer_id',
    how='left'
)


# -------------------------------------------------
# 11) تنظيف عدد عمليات شراء الباقات
# -------------------------------------------------

final_marketing[
    [
        'Known Package Purchase Orders',
        'Unknown Package Purchase Orders'
    ]
] = (
    final_marketing[
        [
            'Known Package Purchase Orders',
            'Unknown Package Purchase Orders'
        ]
    ]
    .fillna(0)
    .astype(int)
)


# -------------------------------------------------
# 12) تحديد الحالة الصحيحة للباقة السابقة
# -------------------------------------------------

# العميل لم يستخدم أي باقة سابقًا
final_marketing.loc[
    ~final_marketing[
        'Has Used Package Before'
    ],
    'Last Known Package'
] = 'لم يستخدم باقة سابقًا'


# العميل استخدم باقة سابقًا،
# لكن لم نستطع تحديد نوعها من مبلغ الشراء
final_marketing.loc[
    (
        final_marketing[
            'Has Used Package Before'
        ]
    )
    &
    (
        final_marketing[
            'Last Known Package'
        ].isna()
    ),
    'Last Known Package'
] = 'استخدم باقة سابقًا - النوع غير محدد'


# -------------------------------------------------
# 13) فحوصات النتيجة
# -------------------------------------------------

print(
    "Package purchase rows by identified package:"
)

display(
    package_purchase_detail[
        'Previous Package Type'
    ]
    .value_counts()
)


print("\nLast known package for all customers:")

display(
    final_marketing[
        'Last Known Package'
    ]
    .value_counts()
)


print("\nPrevious package users only:")

display(
    final_marketing.loc[
        final_marketing[
            'Has Used Package Before'
        ],
        'Last Known Package'
    ]
    .value_counts()
)


In [ ]:
# =========================
# 7.8) Correct days since last order
# =========================

# آخر يوم موجود في بيانات الطلبات، بدون الساعة
analysis_reference_date = (
    pd.to_datetime(
        order_features['order_date'],
        errors='coerce'
    )
    .max()
    .normalize()
)


# تاريخ آخر طلب لكل عميل، بدون الساعة
final_marketing['Last Order Date'] = (
    pd.to_datetime(
        final_marketing['order_date'],
        errors='coerce'
    )
    .dt.normalize()
)


# عدد الأيام منذ آخر طلب
final_marketing['Days Since Last Order'] = (
    analysis_reference_date
    -
    final_marketing['Last Order Date']
).dt.days


# حماية من أي قيمة سالبة غير متوقعة
final_marketing['Days Since Last Order'] = (
    final_marketing['Days Since Last Order']
    .clip(lower=0)
)


# فحص النتيجة
print(
    "Analysis reference date:",
    analysis_reference_date.date()
)

print(
    "Minimum Days Since Last Order:",
    final_marketing[
        'Days Since Last Order'
    ].min()
)

print(
    "Negative values:",
    (
        final_marketing[
            'Days Since Last Order'
        ] < 0
    ).sum()
)


In [ ]:
# =========================
# 7.9) Recent customer spend for package analysis
# =========================

recent_spend_base = order_features[[
    'customer_id',
    'order_id',
    'order_date',
    'order_revenue'
]].copy()


# -------------------------------------------------
# 1) تنظيف التاريخ والإيراد
# -------------------------------------------------

recent_spend_base['order_day'] = (
    pd.to_datetime(
        recent_spend_base['order_date'],
        errors='coerce'
    )
    .dt.normalize()
)

recent_spend_base['order_revenue'] = (
    pd.to_numeric(
        recent_spend_base['order_revenue'],
        errors='coerce'
    )
    .fillna(0)
    .clip(lower=0)
)


# -------------------------------------------------
# 2) تحديد آخر 90 يومًا
# -------------------------------------------------

recent_spend_start_date = (
    analysis_reference_date
    - pd.Timedelta(days=89)
)

recent_spend_mask = (
    (
        recent_spend_base['order_day']
        >= recent_spend_start_date
    )
    &
    (
        recent_spend_base['order_day']
        <= analysis_reference_date
    )
)


# -------------------------------------------------
# 3) تجميع الإنفاق الحديث لكل عميل
# -------------------------------------------------

customer_recent_spend = (
    recent_spend_base.loc[
        recent_spend_mask
    ]
    .groupby('customer_id')
    .agg(
        recent_90d_spend=(
            'order_revenue',
            'sum'
        )
    )
    .reset_index()
)


# -------------------------------------------------
# 4) حساب متوسط الإنفاق الشهري الحديث
# -------------------------------------------------

customer_recent_spend[
    'Recent Monthly Spend Average'
] = (
    customer_recent_spend[
        'recent_90d_spend'
    ] / 3
)


# -------------------------------------------------
# 5) حذف الأعمدة القديمة إذا أعدنا تشغيل البلوك
# -------------------------------------------------

recent_spend_columns = [
    'recent_90d_spend',
    'Recent Monthly Spend Average'
]

existing_recent_spend_columns = [
    column
    for column in recent_spend_columns
    if column in final_marketing.columns
]

if existing_recent_spend_columns:

    final_marketing = final_marketing.drop(
        columns=existing_recent_spend_columns
    )


# -------------------------------------------------
# 6) دمج الإنفاق الحديث مع final_marketing
# -------------------------------------------------

final_marketing = final_marketing.merge(
    customer_recent_spend,
    on='customer_id',
    how='left'
)

final_marketing[
    recent_spend_columns
] = (
    final_marketing[
        recent_spend_columns
    ]
    .fillna(0)
)


# -------------------------------------------------
# 7) فحص النتيجة
# -------------------------------------------------

print(
    "Recent spend period:",
    recent_spend_start_date.date(),
    "to",
    analysis_reference_date.date()
)

display(
    final_marketing[[
        'customer_id',
        'recent_90d_orders',
        'recent_90d_spend',
        'Recent Monthly Spend Average',
        'Recent Monthly Unlimited Laundry Average',
        'Recent Monthly Blanket Average',
        'Recent Monthly Shoe Average'
    ]].head(10)
)


## 12. Dynamic Package Recommendation

Combine behavioral signals, recent usage, prior package history, and eligibility rules into a final recommendation action.


In [ ]:
# =========================
# 7.10) Final dynamic package recommendation logic
# =========================


# -------------------------------------------------
# 1) تنظيف أعمدة الاستخدام المطلوبة
# -------------------------------------------------

package_recommendation_numeric_columns = [
    'recent_90d_orders',
    'recent_90d_unlimited_laundry_quantity',
    'recent_90d_blanket_benefit_quantity',
    'recent_90d_shoe_benefit_quantity',
    'recent_90d_online_orders',

    'Recent Monthly Order Average',
    'Recent Monthly Unlimited Laundry Average',
    'Recent Monthly Blanket Average',
    'Recent Monthly Shoe Average'
]


for column in package_recommendation_numeric_columns:

    final_marketing[column] = (
        pd.to_numeric(
            final_marketing[column],
            errors='coerce'
        )
        .fillna(0)
        .clip(lower=0)
    )


# -------------------------------------------------
# 2) إجمالي الاستخدام الحديث المشمول في الباقات
# -------------------------------------------------

final_marketing['Recent Package Covered Quantity'] = (
    final_marketing[
        'recent_90d_unlimited_laundry_quantity'
    ]
    +
    final_marketing[
        'recent_90d_blanket_benefit_quantity'
    ]
    +
    final_marketing[
        'recent_90d_shoe_benefit_quantity'
    ]
)


# -------------------------------------------------
# 3) قياس اعتماد العميل على الأونلاين والتوصيل
# -------------------------------------------------

final_marketing['Recent Online Order Ratio'] = np.where(
    final_marketing['recent_90d_orders'] > 0,

    final_marketing['recent_90d_online_orders']
    /
    final_marketing['recent_90d_orders'],

    0
)


# طلب أونلاين واحد لا يكفي لترشيح 299.
# نعتبر التوصيل حاجة واضحة فقط إذا:
# - لديه طلبان أونلاين على الأقل.
# - وتشكل طلباته الأونلاين نصف طلباته الحديثة على الأقل.

final_marketing['Strong Recent Online Preference'] = (
    (
        final_marketing[
            'recent_90d_online_orders'
        ] >= 2
    )
    &
    (
        final_marketing[
            'Recent Online Order Ratio'
        ] >= 0.50
    )
)


# -------------------------------------------------
# 4) تجهيز جمهور أول اشتراك لحساب الحدود الديناميكية
# -------------------------------------------------

first_subscription_dynamic_base = final_marketing[
    (
        final_marketing[
            'Package Audience Status'
        ]
        ==
        'لم يستخدم باقة سابقًا - جمهور أول اشتراك'
    )
    &
    (
        final_marketing[
            'recent_90d_orders'
        ] > 0
    )
    &
    (
        final_marketing[
            'Recent Package Covered Quantity'
        ] > 0
    )
].copy()


# -------------------------------------------------
# 5) دالة حساب الحدود الديناميكية
# -------------------------------------------------

def dynamic_count_threshold(
    series,
    quantile_value=0.75,
    minimum_value=1
):

    values = (
        pd.to_numeric(
            series,
            errors='coerce'
        )
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    # نحسب الحد من القيم الموجبة فقط
    values = values[
        values > 0
    ]

    if values.empty:
        return int(
            minimum_value
        )

    threshold = int(
        np.ceil(
            values.quantile(
                quantile_value
            )
        )
    )

    return max(
        int(minimum_value),
        threshold
    )


# -------------------------------------------------
# 6) حدود أهلية أول اشتراك
# تتغير تلقائيًا عند إضافة بيانات جديدة
# -------------------------------------------------

FIRST_SUBSCRIPTION_MIN_RECENT_ORDERS = (
    dynamic_count_threshold(
        first_subscription_dynamic_base[
            'recent_90d_orders'
        ],
        quantile_value=0.75,
        minimum_value=2
    )
)


FIRST_SUBSCRIPTION_MIN_UNLIMITED_QTY_90D = (
    dynamic_count_threshold(
        first_subscription_dynamic_base[
            'recent_90d_unlimited_laundry_quantity'
        ],
        quantile_value=0.75,
        minimum_value=1
    )
)


FIRST_SUBSCRIPTION_MIN_BLANKET_QTY_90D = (
    dynamic_count_threshold(
        first_subscription_dynamic_base[
            'recent_90d_blanket_benefit_quantity'
        ],
        quantile_value=0.75,
        minimum_value=1
    )
)


FIRST_SUBSCRIPTION_MIN_SHOE_QTY_90D = (
    dynamic_count_threshold(
        first_subscription_dynamic_base[
            'recent_90d_shoe_benefit_quantity'
        ],
        quantile_value=0.75,
        minimum_value=1
    )
)


# -------------------------------------------------
# 7) طباعة الحدود المستخدمة في التشغيل الحالي
# -------------------------------------------------

print("Dynamic package eligibility thresholds:")

print(
    "Minimum recent orders:",
    FIRST_SUBSCRIPTION_MIN_RECENT_ORDERS
)

print(
    "Minimum unlimited laundry quantity in 90 days:",
    FIRST_SUBSCRIPTION_MIN_UNLIMITED_QTY_90D
)

print(
    "Minimum blanket quantity in 90 days:",
    FIRST_SUBSCRIPTION_MIN_BLANKET_QTY_90D
)

print(
    "Minimum shoe quantity in 90 days:",
    FIRST_SUBSCRIPTION_MIN_SHOE_QTY_90D
)


# -------------------------------------------------
# 8) تحديد أهلية عميل أول اشتراك
# -------------------------------------------------

first_subscription_mask = (
    final_marketing[
        'Package Audience Status'
    ]
    ==
    'لم يستخدم باقة سابقًا - جمهور أول اشتراك'
)


final_marketing[
    'First Subscription Eligible'
] = False


final_marketing.loc[
    first_subscription_mask,
    'First Subscription Eligible'
] = (
    (
        final_marketing.loc[
            first_subscription_mask,
            'Recent Package Covered Quantity'
        ] > 0
    )
    &
    (
        (
            final_marketing.loc[
                first_subscription_mask,
                'recent_90d_orders'
            ]
            >=
            FIRST_SUBSCRIPTION_MIN_RECENT_ORDERS
        )
        |
        (
            final_marketing.loc[
                first_subscription_mask,
                'recent_90d_unlimited_laundry_quantity'
            ]
            >=
            FIRST_SUBSCRIPTION_MIN_UNLIMITED_QTY_90D
        )
        |
        (
            final_marketing.loc[
                first_subscription_mask,
                'recent_90d_blanket_benefit_quantity'
            ]
            >=
            FIRST_SUBSCRIPTION_MIN_BLANKET_QTY_90D
        )
        |
        (
            final_marketing.loc[
                first_subscription_mask,
                'recent_90d_shoe_benefit_quantity'
            ]
            >=
            FIRST_SUBSCRIPTION_MIN_SHOE_QTY_90D
        )
    )
)


# -------------------------------------------------
# 9) إنشاء سبب أهلية أول اشتراك
# -------------------------------------------------

def build_first_subscription_eligibility_reason(row):

    if not row[
        'First Subscription Eligible'
    ]:
        return (
            'استخدام العميل خلال آخر 90 يومًا '
            ' لا يظهر حاجة كافية للاشتراك حاليًا.'
        )

    eligibility_reasons = []

    recent_orders = int(
        row['recent_90d_orders']
    )

    unlimited_quantity = int(
        row[
            'recent_90d_unlimited_laundry_quantity'
        ]
    )

    blanket_quantity = int(
        row[
            'recent_90d_blanket_benefit_quantity'
        ]
    )

    shoe_quantity = int(
        row[
            'recent_90d_shoe_benefit_quantity'
        ]
    )


    if (
        recent_orders
        >=
        FIRST_SUBSCRIPTION_MIN_RECENT_ORDERS
    ):
        eligibility_reasons.append(
            f'عدد طلباته خلال آخر 90 يومًا هو '
            f'{recent_orders}'
        )


    if (
        unlimited_quantity
        >=
        FIRST_SUBSCRIPTION_MIN_UNLIMITED_QTY_90D
    ):
        eligibility_reasons.append(
            f'إجمالي قطع الغسيل المشمولة خلال '
            f'آخر 90 يومًا هو {unlimited_quantity} قطعة'
        )


    if (
        blanket_quantity
        >=
        FIRST_SUBSCRIPTION_MIN_BLANKET_QTY_90D
    ):
        eligibility_reasons.append(
            f'استخدم {blanket_quantity} بطانيات '
            f'أو ألحفة خلال آخر 90 يومًا'
        )


    if (
        shoe_quantity
        >=
        FIRST_SUBSCRIPTION_MIN_SHOE_QTY_90D
    ):
        eligibility_reasons.append(
            f'استخدم {shoe_quantity} أحذية '
            f'خلال آخر 90 يومًا'
        )


    return '، و'.join(
        eligibility_reasons
    )


final_marketing[
    'First Subscription Eligibility Reason'
] = (
    final_marketing.apply(
        build_first_subscription_eligibility_reason,
        axis=1
    )
)


# -------------------------------------------------
# 10) تحديد الباقة المناسبة حسب الاستخدام الحالي
# -------------------------------------------------

def determine_current_usage_package_fit(row):

    monthly_orders = float(
        row[
            'Recent Monthly Order Average'
        ]
    )

    monthly_blankets = float(
        row[
            'Recent Monthly Blanket Average'
        ]
    )

    monthly_shoes = float(
        row[
            'Recent Monthly Shoe Average'
        ]
    )

    online_orders = int(
        row[
            'recent_90d_online_orders'
        ]
    )

    recent_orders = int(
        row[
            'recent_90d_orders'
        ]
    )

    online_ratio = float(
        row[
            'Recent Online Order Ratio'
        ]
    )

    strong_online_preference = bool(
        row[
            'Strong Recent Online Preference'
        ]
    )


    # -------------------------------------------------
    # Package C
    #
    # نرشحها إذا تجاوز استخدام العميل حدود باقة 299:
    # - أكثر من 4 زيارات شهريًا.
    # - أو أكثر من بطانيتين شهريًا.
    # - أو أكثر من 3 أحذية شهريًا.
    # -------------------------------------------------

    family_reasons = []

    if monthly_orders > 4:
        family_reasons.append(
            f'متوسط زياراته {monthly_orders:.2f} شهريًا، '
            f'وهو أعلى من حد 4 زيارات في باقة 299'
        )

    if monthly_blankets > 2:
        family_reasons.append(
            f'متوسط استخدامه للبطانيات أو الألحفة '
            f'{monthly_blankets:.2f} شهريًا، '
            f'وهو أعلى من حد بطانيتين في باقة 299'
        )

    if monthly_shoes > 3:
        family_reasons.append(
            f'متوسط استخدامه للأحذية '
            f'{monthly_shoes:.2f} شهريًا، '
            f'وهو أعلى من حد 3 أحذية في باقة 299'
        )


    if family_reasons:

        return pd.Series({
            'Current Usage Package Fit':
                'Package C',

            'Current Usage Package Fit Reason':
                '؛ '.join(
                    family_reasons
                )
        })


    # -------------------------------------------------
    # Package B
    #
    # نرشحها إذا:
    # - يعتمد بوضوح على التوصيل المنزلي.
    # - أو تجاوز بطانية واحدة في 199.
    # - أو تجاوز حذاءين في 199.
    #
    # طلب أونلاين واحد وحده لا يكفي.
    # -------------------------------------------------

    youth_plus_reasons = []


    if strong_online_preference:

        youth_plus_reasons.append(
            f'{online_orders} من أصل {recent_orders} '
            f'طلبات حديثة كانت أونلاين '
            f'بنسبة {online_ratio * 100:.0f}%، '
            f'مما يدل على اعتماده على التوصيل المنزلي'
        )


    if monthly_blankets > 1:

        youth_plus_reasons.append(
            f'متوسط استخدامه للبطانيات أو الألحفة '
            f'{monthly_blankets:.2f} شهريًا، '
            f'وهو أعلى من حد بطانية واحدة في باقة 199'
        )


    if monthly_shoes > 2:

        youth_plus_reasons.append(
            f'متوسط استخدامه للأحذية '
            f'{monthly_shoes:.2f} شهريًا، '
            f'وهو أعلى من حد حذاءين في باقة 199'
        )


    if youth_plus_reasons:

        return pd.Series({
            'Current Usage Package Fit':
                'Package B',

            'Current Usage Package Fit Reason':
                '؛ '.join(
                    youth_plus_reasons
                )
        })


    # -------------------------------------------------
    # Package A
    #
    # العميل مؤهل للاشتراك، لكن استخدامه لا يحتاج
    # مزايا 299 أو 399.
    # -------------------------------------------------

    return pd.Series({
        'Current Usage Package Fit':
            'Package A',

        'Current Usage Package Fit Reason':
            (
                'استخدامه الشهري لا يتجاوز 4 زيارات، '
                'وبطانية واحدة، وحذاءين، '
                'ولا يوجد اعتماد واضح ومتكرر '
                'على التوصيل المنزلي'
            )
    })


# حذف الأعمدة القديمة إذا أعدنا تشغيل البلوك
current_usage_fit_columns = [
    'Current Usage Package Fit',
    'Current Usage Package Fit Reason'
]


existing_current_usage_fit_columns = [
    column
    for column in current_usage_fit_columns
    if column in final_marketing.columns
]


if existing_current_usage_fit_columns:

    final_marketing = final_marketing.drop(
        columns=existing_current_usage_fit_columns
    )


current_usage_fit_result = (
    final_marketing.apply(
        determine_current_usage_package_fit,
        axis=1
    )
)


final_marketing = pd.concat(
    [
        final_marketing,
        current_usage_fit_result
    ],
    axis=1
)


# -------------------------------------------------
# 11) ترتيب الباقات للمقارنة عند التجديد
# -------------------------------------------------

package_rank = {
    'Package A': 1,
    'Package B': 2,
    'Package C': 3
}


# -------------------------------------------------
# 12) إنشاء التوصية النهائية والسبب الكامل
# -------------------------------------------------

def final_package_recommendation(row):

    audience_status = row[
        'Package Audience Status'
    ]

    previous_package = row[
        'Last Known Package'
    ]

    current_fit = row[
        'Current Usage Package Fit'
    ]

    current_fit_reason = row[
        'Current Usage Package Fit Reason'
    ]

    recent_orders = int(
        row[
            'recent_90d_orders'
        ]
    )

    recent_covered_usage = float(
        row[
            'Recent Package Covered Quantity'
        ]
    )


    # -------------------------------------------------
    # الحسابات غير الفردية
    # -------------------------------------------------

    if (
        audience_status
        ==
        'حساب بطلبات يومية غير طبيعية'
    ):

        return pd.Series({
            'Package Recommendation Category':
                'يحتاج مراجعة',

            'Package Recommendation Action':
                'مراجعة الحساب قبل الترشيح',

            'Recommended Package':
                'لا توجد توصية آلية',

            'Package Recommendation Reason':
                (
                    'الحساب لديه نمط طلبات يومية غير طبيعي، '
                    'لذلك لا يدخل ضمن ترشيحات باقات الأفراد '
                    'قبل مراجعته.'
                )
        })


    # -------------------------------------------------
    # المشترك الحالي
    # -------------------------------------------------

    if (
        audience_status
        ==
        'مشترك حاليًا - لا يعرض عليه اشتراك جديد'
    ):

        return pd.Series({
            'Package Recommendation Category':
                'لا يوجد عرض حاليًا',

            'Package Recommendation Action':
                'مشترك حاليًا - بدون عرض جديد',

            'Recommended Package':
                'مشترك حاليًا',

            'Package Recommendation Reason':
                (
                    'العميل لديه اشتراك فعال حاليًا، '
                    'لذلك لا يُعرض عليه اشتراك جديد.'
                )
        })


    # -------------------------------------------------
    # جمهور أول اشتراك
    # -------------------------------------------------

    if (
        audience_status
        ==
        'لم يستخدم باقة سابقًا - جمهور أول اشتراك'
    ):

        if not row[
            'First Subscription Eligible'
        ]:

            return pd.Series({
                'Package Recommendation Category':
                    'لا يوجد عرض حاليًا',

                'Package Recommendation Action':
                    'لا يُعرض عليه اشتراك حاليًا',

                'Recommended Package':
                    'لا توجد باقة حاليًا',

                'Package Recommendation Reason':
                    (
                        'استخدام العميل خلال آخر 90 يومًا '
                        'لا يظهر حاجة كافية للاشتراك, '
                        'لذلك لا يُعرض عليه اشتراك حاليًا.'
                    )
            })


        eligibility_reason = row[
            'First Subscription Eligibility Reason'
        ]


        return pd.Series({
            'Package Recommendation Category':
                'أول اشتراك',

            'Package Recommendation Action':
                'عرض اشتراك لأول مرة',

            'Recommended Package':
                current_fit,

            'Package Recommendation Reason':
                (
                    f'أصبح العميل مؤهلًا للاشتراك لأن '
                    f'{eligibility_reason}. '
                    f'تم ترشيح {current_fit} لأن '
                    f'{current_fit_reason}.'
                )
        })


    # -------------------------------------------------
    # جمهور التجديد
    # -------------------------------------------------

    if (
        audience_status
        ==
        'استخدم باقة سابقًا - جمهور التجديد'
    ):

        previous_package_is_known = (
            previous_package
            in
            package_rank
        )


        # لا يوجد استخدام حديث
        if recent_orders == 0:

            if previous_package_is_known:

                return pd.Series({
                    'Package Recommendation Category':
                        'تجديد اشتراك',

                    'Package Recommendation Action':
                        'استرجاع العميل وتجديد الباقة السابقة',

                    'Recommended Package':
                        previous_package,

                    'Package Recommendation Reason':
                        (
                            f'العميل اشترك سابقًا في '
                            f'{previous_package}، '
                            f'ولا يوجد له نشاط خلال آخر 90 يومًا. '
                            f'لذلك تكون التوصية استرجاع العميل '
                            f'وعرض تجديد باقته السابقة، '
                            f'بدون ترقية غير مبنية على استخدام حديث.'
                        )
                })


            return pd.Series({
                'Package Recommendation Category':
                    'يحتاج مراجعة',

                'Package Recommendation Action':
                    'تجديد - بيانات الباقة السابقة غير كافية',

                'Recommended Package':
                    'الباقة السابقة غير محددة',

                'Package Recommendation Reason':
                    (
                        'العميل استخدم باقة سابقًا، '
                        'لكن نوع الباقة السابقة غير معروف، '
                        'ولا يوجد استخدام حديث يسمح '
                        'باختيار باقة مناسبة آليًا.'
                    )
            })


        # عنده نشاط حديث لكنه غير مرتبط بمزايا الباقات
        if recent_covered_usage == 0:

            if previous_package_is_known:

                return pd.Series({
                    'Package Recommendation Category':
                        'تجديد اشتراك',

                    'Package Recommendation Action':
                        'تجديد الباقة السابقة',

                    'Recommended Package':
                        previous_package,

                    'Package Recommendation Reason':
                        (
                            f'آخر باقة معروفة للعميل هي '
                            f'{previous_package}. '
                            f'لديه نشاط حديث، لكن الطلبات المسجلة '
                            f'لا تحتوي استخدامًا واضحًا لمزايا '
                            f'الغسيل أو البطانيات أو الأحذية؛ '
                            f'لذلك نحافظ على الباقة السابقة '
                            f'بدون ترقية آلية.'
                        )
                })


            return pd.Series({
                'Package Recommendation Category':
                    'يحتاج مراجعة',

                'Package Recommendation Action':
                    'تجديد - بيانات الباقة السابقة غير كافية',

                'Recommended Package':
                    'الباقة السابقة غير محددة',

                'Package Recommendation Reason':
                    (
                        'نوع الباقة السابقة غير معروف، '
                        'والاستخدام الحديث لا يحتوي مؤشرات '
                        'واضحة تسمح باختيار باقة آليًا.'
                    )
            })


        # الباقة السابقة معروفة
        if previous_package_is_known:

            previous_rank = package_rank[
                previous_package
            ]

            current_fit_rank = package_rank[
                current_fit
            ]


            # نرقي فقط إذا الاستخدام الحالي يتطلب باقة أعلى
            if current_fit_rank > previous_rank:

                return pd.Series({
                    'Package Recommendation Category':
                        'تجديد اشتراك',

                    'Package Recommendation Action':
                        'تجديد مع ترقية إلى باقة أعلى',

                    'Recommended Package':
                        current_fit,

                    'Package Recommendation Reason':
                        (
                            f'العميل كان مشتركًا في '
                            f'{previous_package}. '
                            f'استخدامه الحالي يشير إلى أن '
                            f'{current_fit_reason}. '
                            f'لذلك تم ترشيح {current_fit} '
                            f'عند التجديد.'
                        )
                })


            # لا نخفض الباقة تلقائيًا
            return pd.Series({
                'Package Recommendation Category':
                    'تجديد اشتراك',

                'Package Recommendation Action':
                    'تجديد الباقة السابقة',

                'Recommended Package':
                    previous_package,

                'Package Recommendation Reason':
                    (
                        f'آخر باقة معروفة للعميل هي '
                        f'{previous_package}. '
                        f'استخدامه الحالي لا يتطلب الانتقال '
                        f'إلى باقة أعلى، لذلك نوصي بتجديد '
                        f'الباقة السابقة. '
                        f'ولا يتم تخفيض الباقة تلقائيًا '
                        f'اعتمادًا على فترة استخدام قصيرة.'
                    )
            })


        # نوع الباقة السابقة غير معروف،
        # لكن يوجد استخدام حديث يسمح بالاختيار
        return pd.Series({
            'Package Recommendation Category':
                'تجديد اشتراك',

            'Package Recommendation Action':
                'تجديد وتحديد الباقة حسب الاستخدام الحالي',

            'Recommended Package':
                current_fit,

            'Package Recommendation Reason':
                (
                    f'نوع الباقة السابقة غير معروف، '
                    f'لكن الاستخدام الحالي يوضح أن '
                    f'{current_fit_reason}. '
                    f'لذلك تم ترشيح {current_fit}.'
                )
        })


    # -------------------------------------------------
    # حالة احتياطية
    # -------------------------------------------------

    return pd.Series({
        'Package Recommendation Category':
            'يحتاج مراجعة',

        'Package Recommendation Action':
            'مراجعة الحساب قبل الترشيح',

        'Recommended Package':
            'لا توجد توصية آلية',

        'Package Recommendation Reason':
            (
                'حالة العميل لا تطابق قواعد '
                'الترشيح الحالية.'
            )
    })


# -------------------------------------------------
# 13) حذف نتائج الترشيح القديمة
# -------------------------------------------------

recommendation_output_columns = [
    'Package Recommendation Category',
    'Package Recommendation Action',
    'Recommended Package',
    'Package Recommendation Reason'
]


existing_recommendation_columns = [
    column
    for column in recommendation_output_columns
    if column in final_marketing.columns
]


if existing_recommendation_columns:

    final_marketing = final_marketing.drop(
        columns=existing_recommendation_columns
    )


# -------------------------------------------------
# 14) تطبيق التوصية النهائية
# -------------------------------------------------

package_recommendation_result = (
    final_marketing.apply(
        final_package_recommendation,
        axis=1
    )
)


final_marketing = pd.concat(
    [
        final_marketing,
        package_recommendation_result
    ],
    axis=1
)


# -------------------------------------------------
# 15) فحوصات النتيجة
# -------------------------------------------------

print("\nPackage recommendation categories:")

display(
    final_marketing[
        'Package Recommendation Category'
    ].value_counts()
)


print("\nPackage recommendation actions:")

display(
    final_marketing[
        'Package Recommendation Action'
    ].value_counts()
)


print("\nRecommended packages:")

display(
    final_marketing[
        'Recommended Package'
    ].value_counts()
)


print("\nSample recommendations with full reasons:")

with pd.option_context(
    'display.max_colwidth',
    None
):

    display(
        final_marketing.loc[
            final_marketing[
                'Package Recommendation Category'
            ].isin([
                'أول اشتراك',
                'تجديد اشتراك'
            ]),
            [
                'customer_id',
                'Package Recommendation Category',
                'Package Recommendation Action',
                'Last Known Package',
                'Recommended Package',
                'Package Recommendation Reason'
            ]
        ].head(10)
    )


# ☎️ AI Voice Calling Prioritization Layer

Convert ML and marketing outputs into an actionable outbound call queue for the AI voice sales system.


In [ ]:
# =========================================================
# 18) Calling Layer
# =========================================================


# =========================
# 18.1) Call decision
# =========================

# -------------------------------------------------
# 1) التأكد من وجود الأعمدة المطلوبة
# -------------------------------------------------

required_calling_columns = [
    'customer_phone',
    'Package Recommendation Category',
    'Package Recommendation Action',
    'Recommended Package',
    'Package Recommendation Reason'
]


missing_calling_columns = [
    column
    for column in required_calling_columns
    if column not in final_marketing.columns
]


if missing_calling_columns:

    raise KeyError(
        f'Missing columns required for Calling Layer: '
        f'{missing_calling_columns}'
    )


# -------------------------------------------------
# 2) إنشاء جدول مستقل لطبقة الاتصال
# -------------------------------------------------

# ننسخ جميع قرارات الماركتنق من بلوك 17
# ثم نضيف عليها قرارات الاتصال
calling_layer = final_marketing.copy()


# -------------------------------------------------
# 3) فحص توفر رقم هاتف صالح مبدئيًا
# -------------------------------------------------

call_phone_text = (
    calling_layer[
        'customer_phone'
    ]
    .fillna('')
    .astype(str)
    .str.strip()
)


call_phone_digits = (
    call_phone_text
    .str.replace(
        r'\D',
        '',
        regex=True
    )
)


# نعتبر الرقم متوفرًا مبدئيًا إذا كان
# يحتوي من 8 إلى 15 رقمًا
calling_layer['Has Callable Phone'] = (
    call_phone_digits
    .str.len()
    .between(
        8,
        15
    )
)


# -------------------------------------------------
# 4) تحديد قرار الاتصال لكل عميل
# -------------------------------------------------

def determine_call_decision(row):

    has_phone = row[
        'Has Callable Phone'
    ]

    package_category = row[
        'Package Recommendation Category'
    ]

    package_action = row[
        'Package Recommendation Action'
    ]

    recommended_package = row[
        'Recommended Package'
    ]

    package_reason = row[
        'Package Recommendation Reason'
    ]


    # ---------------------------------------------
    # لا يوجد رقم صالح للاتصال
    # ---------------------------------------------

    if not has_phone:

        return pd.Series({
            'Should Call':
                False,

            'Call Decision':
                'لا تتصل حاليًا',

            'Call Decision Reason':
                'لا يوجد رقم هاتف صالح ومتاح للاتصال.'
        })


    # ---------------------------------------------
    # عميل مؤهل لأول اشتراك
    # ---------------------------------------------

    if package_category == 'أول اشتراك':

        return pd.Series({
            'Should Call':
                True,

            'Call Decision':
                'اتصل بالعميل',

            'Call Decision Reason':
                (
                    f'العميل مؤهل لعرض اشتراك لأول مرة، '
                    f'والباقة المقترحة له هي '
                    f'{recommended_package}.'
                )
        })


    # ---------------------------------------------
    # عميل تجديد اشتراك
    # ---------------------------------------------

    if package_category == 'تجديد اشتراك':

        return pd.Series({
            'Should Call':
                True,

            'Call Decision':
                'اتصل بالعميل',

            'Call Decision Reason':
                (
                    f'العميل من جمهور التجديد. '
                    f'الإجراء المطلوب هو: '
                    f'{package_action}. '
                    f'الباقة المقترحة: '
                    f'{recommended_package}.'
                )
        })


    # ---------------------------------------------
    # لا يوجد عرض حاليًا
    # ---------------------------------------------

    if package_category == 'لا يوجد عرض حاليًا':

        if (
            package_action
            ==
            'مشترك حاليًا - بدون عرض جديد'
        ):

            return pd.Series({
                'Should Call':
                    False,

                'Call Decision':
                    'لا تتصل حاليًا',

                'Call Decision Reason':
                    (
                        'العميل لديه اشتراك فعال حاليًا، '
                        'ولا يوجد عرض اشتراك جديد له.'
                    )
            })


        return pd.Series({
            'Should Call':
                False,

            'Call Decision':
                'لا تتصل حاليًا',

            'Call Decision Reason':
                (
                    'العميل غير مؤهل حاليًا '
                    'لعرض اشتراك حسب استخدامه الحديث.'
                )
        })


    # ---------------------------------------------
    # حالات تحتاج مراجعة
    # ---------------------------------------------

    if package_category == 'يحتاج مراجعة':

        return pd.Series({
            'Should Call':
                False,

            'Call Decision':
                'مراجعة قبل الاتصال',

            'Call Decision Reason':
                package_reason
        })


    # ---------------------------------------------
    # حالة غير متوقعة
    # ---------------------------------------------

    return pd.Series({
        'Should Call':
            False,

        'Call Decision':
            'مراجعة قبل الاتصال',

        'Call Decision Reason':
            (
                'حالة العميل لا تطابق '
                'قواعد الاتصال الحالية.'
            )
    })


# -------------------------------------------------
# 5) تطبيق قرار الاتصال
# -------------------------------------------------

call_decision_result = (
    calling_layer.apply(
        determine_call_decision,
        axis=1
    )
)


# حماية عند إعادة تشغيل الجزء
call_decision_columns = [
    'Should Call',
    'Call Decision',
    'Call Decision Reason'
]


existing_call_decision_columns = [
    column
    for column in call_decision_columns
    if column in calling_layer.columns
]


if existing_call_decision_columns:

    calling_layer = calling_layer.drop(
        columns=existing_call_decision_columns
    )


calling_layer = pd.concat(
    [
        calling_layer,
        call_decision_result
    ],
    axis=1
)


# -------------------------------------------------
# 6) فحوصات النتيجة
# -------------------------------------------------

print("Call decisions:")

display(
    calling_layer[
        'Call Decision'
    ].value_counts()
)


print("\nShould call distribution:")

display(
    calling_layer[
        'Should Call'
    ].value_counts()
)


print("\nCall decision by package category:")

display(
    pd.crosstab(
        calling_layer[
            'Package Recommendation Category'
        ],
        calling_layer[
            'Call Decision'
        ]
    )
)


## 13. Call Objective Selection

Assign a clear objective for each recommended call, such as acquisition, renewal, reactivation, or upgrade.


In [ ]:
# =========================
# 18.2) Call objective
# =========================

# -------------------------------------------------
# 1) تحديد هدف المكالمة لكل عميل
# -------------------------------------------------

def determine_call_objective(row):

    should_call = row[
        'Should Call'
    ]

    call_decision = row[
        'Call Decision'
    ]

    package_action = row[
        'Package Recommendation Action'
    ]

    recommended_package = row[
        'Recommended Package'
    ]

    previous_package = row[
        'Last Known Package'
    ]


    # ---------------------------------------------
    # لا يوجد اتصال
    # ---------------------------------------------

    if not should_call:

        if call_decision == 'مراجعة قبل الاتصال':

            return pd.Series({
                'Call Objective':
                    'لا توجد مكالمة قبل المراجعة',

                'Call Objective Details':
                    (
                        'يجب مراجعة بيانات العميل '
                        'قبل إضافته إلى قائمة الاتصال.'
                    )
            })


        return pd.Series({
            'Call Objective':
                'لا توجد مكالمة حاليًا',

            'Call Objective Details':
                (
                    'العميل غير مدرج حاليًا '
                    'ضمن قائمة الاتصال.'
                )
        })


    # ---------------------------------------------
    # عرض اشتراك لأول مرة
    # ---------------------------------------------

    if (
        package_action
        ==
        'عرض اشتراك لأول مرة'
    ):

        return pd.Series({
            'Call Objective':
                'بيع اشتراك لأول مرة',

            'Call Objective Details':
                (
                    f'التواصل مع العميل لإقناعه '
                    f'بالاشتراك لأول مرة في '
                    f'{recommended_package}.'
                )
        })


    # ---------------------------------------------
    # تجديد الباقة السابقة
    # ---------------------------------------------

    if (
        package_action
        ==
        'تجديد الباقة السابقة'
    ):

        return pd.Series({
            'Call Objective':
                'تجديد الباقة السابقة',

            'Call Objective Details':
                (
                    f'التواصل مع العميل لتجديد '
                    f'{recommended_package}، '
                    f'وهي آخر باقة مناسبة ومعروفة له.'
                )
        })


    # ---------------------------------------------
    # تجديد مع ترقية
    # ---------------------------------------------

    if (
        package_action
        ==
        'تجديد مع ترقية إلى باقة أعلى'
    ):

        return pd.Series({
            'Call Objective':
                'تجديد الاشتراك مع ترقية',

            'Call Objective Details':
                (
                    f'التواصل مع العميل لتجديد اشتراكه '
                    f'مع الترقية من {previous_package} '
                    f'إلى {recommended_package} '
                    f'بناءً على استخدامه الحالي.'
                )
        })


    # ---------------------------------------------
    # استرجاع العميل وتجديد الباقة
    # ---------------------------------------------

    if (
        package_action
        ==
        'استرجاع العميل وتجديد الباقة السابقة'
    ):

        return pd.Series({
            'Call Objective':
                'استرجاع عميل وتجديد الاشتراك',

            'Call Objective Details':
                (
                    f'إعادة تنشيط العميل وإقناعه '
                    f'بتجديد {recommended_package}، '
                    f'وهي آخر باقة اشترك بها.'
                )
        })


    # ---------------------------------------------
    # تجديد حسب الاستخدام الحالي
    # ---------------------------------------------

    if (
        package_action
        ==
        'تجديد وتحديد الباقة حسب الاستخدام الحالي'
    ):

        return pd.Series({
            'Call Objective':
                'تجديد حسب الاستخدام الحالي',

            'Call Objective Details':
                (
                    f'نوع الباقة السابقة غير معروف، '
                    f'والهدف هو تجديد اشتراك العميل '
                    f'بعرض {recommended_package} '
                    f'حسب استخدامه الحالي.'
                )
        })


    # ---------------------------------------------
    # حالة احتياطية
    # ---------------------------------------------

    return pd.Series({
        'Call Objective':
            'مراجعة هدف المكالمة',

        'Call Objective Details':
            (
                'العميل محدد للاتصال، لكن هدف '
                'المكالمة يحتاج مراجعة.'
            )
    })


# -------------------------------------------------
# 2) حذف النتائج القديمة عند إعادة التشغيل
# -------------------------------------------------

call_objective_columns = [
    'Call Objective',
    'Call Objective Details'
]


existing_call_objective_columns = [
    column
    for column in call_objective_columns
    if column in calling_layer.columns
]


if existing_call_objective_columns:

    calling_layer = calling_layer.drop(
        columns=existing_call_objective_columns
    )


# -------------------------------------------------
# 3) تطبيق هدف المكالمة
# -------------------------------------------------

call_objective_result = (
    calling_layer.apply(
        determine_call_objective,
        axis=1
    )
)


calling_layer = pd.concat(
    [
        calling_layer,
        call_objective_result
    ],
    axis=1
)


# -------------------------------------------------
# 4) فحوصات النتيجة
# -------------------------------------------------

print("Call objectives:")

display(
    calling_layer[
        'Call Objective'
    ].value_counts()
)


print("\nCall objectives for callable customers only:")

display(
    calling_layer.loc[
        calling_layer[
            'Should Call'
        ],
        'Call Objective'
    ].value_counts()
)


print("\nCall objective by recommended package:")

display(
    pd.crosstab(
        calling_layer.loc[
            calling_layer[
                'Should Call'
            ],
            'Call Objective'
        ],
        calling_layer.loc[
            calling_layer[
                'Should Call'
            ],
            'Recommended Package'
        ]
    )
)


## 14. Call Priority & Queue Ranking

Score callable customers and rank the queue so the outbound automation contacts the highest-value opportunities first.


In [ ]:
# =========================
# 18.3) Call priority and queue ranking
# =========================


# -------------------------------------------------
# 1) الأعمدة المطلوبة
# -------------------------------------------------

required_priority_columns = [
    'Should Call',
    'Call Objective',
    'Package Recommendation Action',
    'Recommended Package',

    'return_probability',
    'Customer Value Segment',
    'Total Spend To Date',
    'Days Since Last Order',

    'recent_90d_orders',
    'Recent Package Covered Quantity'
]


missing_priority_columns = [
    column
    for column in required_priority_columns
    if column not in calling_layer.columns
]


if missing_priority_columns:

    raise KeyError(
        f'Missing columns required for call priority: '
        f'{missing_priority_columns}'
    )


# -------------------------------------------------
# 2) تنظيف الأعمدة الرقمية
# -------------------------------------------------

priority_numeric_columns = [
    'return_probability',
    'Total Spend To Date',
    'Days Since Last Order',
    'recent_90d_orders',
    'Recent Package Covered Quantity'
]


for column in priority_numeric_columns:

    calling_layer[column] = (
        pd.to_numeric(
            calling_layer[column],
            errors='coerce'
        )
        .fillna(0)
    )


calling_layer['return_probability'] = (
    calling_layer[
        'return_probability'
    ]
    .clip(
        lower=0,
        upper=1
    )
)


for column in [
    'Total Spend To Date',
    'Days Since Last Order',
    'recent_90d_orders',
    'Recent Package Covered Quantity'
]:

    calling_layer[column] = (
        calling_layer[column]
        .clip(lower=0)
    )


calling_layer['Should Call'] = (
    calling_layer[
        'Should Call'
    ]
    .fillna(False)
    .astype(bool)
)


callable_mask = (
    calling_layer[
        'Should Call'
    ]
)


# -------------------------------------------------
# 3) دالة تحويل السلوك إلى ترتيب نسبي ديناميكي
# -------------------------------------------------

def dynamic_percentile_score(
    series,
    higher_is_better=True
):

    values = (
        pd.to_numeric(
            series,
            errors='coerce'
        )
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(0)
    )


    if values.empty:
        return pd.Series(
            dtype=float
        )


    # لو كل القيم متساوية، نعطي الجميع نتيجة وسطية
    if values.nunique() <= 1:

        return pd.Series(
            0.50,
            index=values.index,
            dtype=float
        )


    # ascending=True:
    # القيمة الأعلى تحصل على Percentile أعلى
    #
    # ascending=False:
    # القيمة الأقل تحصل على Percentile أعلى
    return values.rank(
        method='average',
        pct=True,
        ascending=higher_is_better
    )


# -------------------------------------------------
# 4) ترتيب قوة هدف المكالمة
# -------------------------------------------------

# هذه النقاط لا تعني أن العميل سيشتري بالتأكيد.
# هي تعبّر عن جاهزية العميل للبيع:
# عميل التجديد أقرب للإغلاق من عميل أول اشتراك.

call_action_readiness_mapping = {

    'تجديد الباقة السابقة':
        1.00,

    'تجديد مع ترقية إلى باقة أعلى':
        0.95,

    'تجديد وتحديد الباقة حسب الاستخدام الحالي':
        0.90,

    'استرجاع العميل وتجديد الباقة السابقة':
        0.80,

    'عرض اشتراك لأول مرة':
        0.70
}


calling_layer[
    'Call Action Readiness Score'
] = (
    calling_layer[
        'Package Recommendation Action'
    ]
    .map(
        call_action_readiness_mapping
    )
    .fillna(0)
)


# -------------------------------------------------
# 5) تقييم قيمة العميل
# -------------------------------------------------

customer_value_score_mapping = {

    'عميل متكرر عالي الإنفاق':
        1.00,

    'عميل متكرر متوسط الإنفاق':
        0.80,

    'عميل جديد - طلب أول مرتفع':
        0.70,

    'عميل متكرر منخفض الإنفاق':
        0.55,

    'عميل جديد - طلب أول عادي':
        0.35,

    'حساب بطلبات يومية غير طبيعية':
        0.00
}


calling_layer[
    'Customer Value Priority Score'
] = (
    calling_layer[
        'Customer Value Segment'
    ]
    .map(
        customer_value_score_mapping
    )
    .fillna(0.50)
)


# -------------------------------------------------
# 6) تقييم القيمة التجارية للباقة
# -------------------------------------------------

package_commercial_value_mapping = {

    'Package C':
        1.00,

    'Package B':
        0.65,

    'Package A':
        0.35
}


calling_layer[
    'Package Commercial Value Score'
] = (
    calling_layer[
        'Recommended Package'
    ]
    .map(
        package_commercial_value_mapping
    )
    .fillna(0)
)


# -------------------------------------------------
# 7) إنشاء درجات ديناميكية داخل العملاء القابلين للاتصال
# -------------------------------------------------

dynamic_priority_columns = [
    'Return Probability Priority Score',
    'Total Spend Priority Score',
    'Recency Priority Score',
    'Recent Orders Priority Score',
    'Recent Usage Priority Score'
]


for column in dynamic_priority_columns:

    calling_layer[column] = 0.0


# احتمالية العودة الأعلى أفضل لفرصة الإغلاق
calling_layer.loc[
    callable_mask,
    'Return Probability Priority Score'
] = (
    dynamic_percentile_score(
        calling_layer.loc[
            callable_mask,
            'return_probability'
        ],
        higher_is_better=True
    )
)


# إجمالي الإنفاق الأعلى يعبر عن قيمة تجارية أعلى
calling_layer.loc[
    callable_mask,
    'Total Spend Priority Score'
] = (
    dynamic_percentile_score(
        calling_layer.loc[
            callable_mask,
            'Total Spend To Date'
        ],
        higher_is_better=True
    )
)


# عدد أيام أقل منذ آخر طلب = عميل أحدث نشاطًا
calling_layer.loc[
    callable_mask,
    'Recency Priority Score'
] = (
    dynamic_percentile_score(
        calling_layer.loc[
            callable_mask,
            'Days Since Last Order'
        ],
        higher_is_better=False
    )
)


# كثرة الطلبات الحديثة ترفع الجاهزية
calling_layer.loc[
    callable_mask,
    'Recent Orders Priority Score'
] = (
    dynamic_percentile_score(
        calling_layer.loc[
            callable_mask,
            'recent_90d_orders'
        ],
        higher_is_better=True
    )
)


# كثرة الاستخدام المشمول في الباقات تدعم الحاجة للاشتراك
calling_layer.loc[
    callable_mask,
    'Recent Usage Priority Score'
] = (
    dynamic_percentile_score(
        calling_layer.loc[
            callable_mask,
            'Recent Package Covered Quantity'
        ],
        higher_is_better=True
    )
)


# -------------------------------------------------
# 8) حساب درجة أولوية الاتصال
# -------------------------------------------------

# توزيع الوزن:
#
# 25% جاهزية هدف المكالمة:
#     التجديد أقرب للإغلاق من أول اشتراك
#
# 20% احتمالية العودة
#
# 20% قيمة العميل:
#     10% شريحة العميل
#     10% ترتيبه الفعلي في الإنفاق
#
# 15% حداثة آخر طلب
#
# 15% النشاط الحديث:
#     7.5% عدد الطلبات
#     7.5% حجم الاستخدام المشمول
#
# 5% القيمة التجارية للباقة

calling_layer[
    'Call Priority Score'
] = 0.0


calling_layer.loc[
    callable_mask,
    'Call Priority Score'
] = (

    calling_layer.loc[
        callable_mask,
        'Call Action Readiness Score'
    ] * 25

    +

    calling_layer.loc[
        callable_mask,
        'Return Probability Priority Score'
    ] * 20

    +

    calling_layer.loc[
        callable_mask,
        'Customer Value Priority Score'
    ] * 10

    +

    calling_layer.loc[
        callable_mask,
        'Total Spend Priority Score'
    ] * 10

    +

    calling_layer.loc[
        callable_mask,
        'Recency Priority Score'
    ] * 15

    +

    calling_layer.loc[
        callable_mask,
        'Recent Orders Priority Score'
    ] * 7.5

    +

    calling_layer.loc[
        callable_mask,
        'Recent Usage Priority Score'
    ] * 7.5

    +

    calling_layer.loc[
        callable_mask,
        'Package Commercial Value Score'
    ] * 5
)


calling_layer[
    'Call Priority Score'
] = (
    calling_layer[
        'Call Priority Score'
    ]
    .round(1)
    .clip(
        lower=0,
        upper=100
    )
)


# -------------------------------------------------
# 9) إنشاء Percentile للأولوية
# -------------------------------------------------

calling_layer[
    'Call Priority Percentile'
] = 0.0


calling_layer.loc[
    callable_mask,
    'Call Priority Percentile'
] = (
    dynamic_percentile_score(
        calling_layer.loc[
            callable_mask,
            'Call Priority Score'
        ],
        higher_is_better=True
    )
)


# -------------------------------------------------
# 10) تقسيم مستويات الأولوية ديناميكيًا
# -------------------------------------------------

def determine_call_priority_level(row):

    if not row[
        'Should Call'
    ]:

        return 'خارج قائمة الاتصال'


    priority_percentile = row[
        'Call Priority Percentile'
    ]


    # أعلى 10%
    if priority_percentile >= 0.90:

        return (
            'أولوية 1 - أعلى فرصة إغلاق'
        )


    # من أعلى 10% إلى أعلى 30%
    if priority_percentile >= 0.70:

        return (
            'أولوية 2 - فرصة إغلاق عالية'
        )


    # من أعلى 30% إلى أعلى 60%
    if priority_percentile >= 0.40:

        return (
            'أولوية 3 - فرصة إغلاق جيدة'
        )


    # آخر 40% من قائمة الاتصال
    return (
        'أولوية 4 - متابعة لاحقة'
    )


calling_layer[
    'Call Priority Level'
] = (
    calling_layer.apply(
        determine_call_priority_level,
        axis=1
    )
)


# -------------------------------------------------
# 11) ترتيب العملاء داخل قائمة الاتصال
# -------------------------------------------------

calling_layer[
    'Call Queue Rank'
] = pd.Series(
    pd.NA,
    index=calling_layer.index,
    dtype='Int64'
)


call_queue_order = (
    calling_layer.loc[
        callable_mask
    ]
    .sort_values(
        by=[
            'Call Priority Score',
            'Call Action Readiness Score',
            'return_probability',
            'Total Spend To Date',
            'Days Since Last Order'
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True
        ],
        kind='stable'
    )
    .index
)


calling_layer.loc[
    call_queue_order,
    'Call Queue Rank'
] = np.arange(
    1,
    len(call_queue_order) + 1
)


# -------------------------------------------------
# 12) إنشاء سبب واضح للأولوية
# -------------------------------------------------

def build_call_priority_reason(row):

    if not row[
        'Should Call'
    ]:

        return (
            'العميل غير مدرج ضمن قائمة الاتصال الحالية.'
        )


    action = row[
        'Package Recommendation Action'
    ]

    value_segment = row[
        'Customer Value Segment'
    ]

    return_probability_percent = (
        row[
            'return_probability'
        ] * 100
    )

    days_since_last_order = int(
        row[
            'Days Since Last Order'
        ]
    )

    recent_orders = int(
        row[
            'recent_90d_orders'
        ]
    )

    recommended_package = row[
        'Recommended Package'
    ]


    return (
        f'تم ترتيب العميل بناءً على أن هدف الاتصال هو '
        f'{action}، '
        f'واحتمالية عودته {return_probability_percent:.1f}%، '
        f'وتصنيف قيمته: {value_segment}، '
        f'وآخر طلب له منذ {days_since_last_order} يومًا، '
        f'ولديه {recent_orders} طلبات خلال آخر 90 يومًا، '
        f'والباقة المقترحة هي {recommended_package}.'
    )


calling_layer[
    'Call Priority Reason'
] = (
    calling_layer.apply(
        build_call_priority_reason,
        axis=1
    )
)


# -------------------------------------------------
# 13) فحوصات النتيجة
# -------------------------------------------------

print("Call priority levels:")

display(
    calling_layer[
        'Call Priority Level'
    ].value_counts()
)


print("\nCallable customers by priority level:")

display(
    calling_layer.loc[
        calling_layer[
            'Should Call'
        ],
        'Call Priority Level'
    ].value_counts()
)


print("\nTop 20 customers in the call queue:")

with pd.option_context(
    'display.max_colwidth',
    None
):

    display(
        calling_layer.loc[
            calling_layer[
                'Should Call'
            ],
            [
                'customer_id',
                'Call Queue Rank',
                'Call Priority Score',
                'Call Priority Level',
                'Call Objective',
                'Recommended Package',
                'Call Priority Reason'
            ]
        ]
        .sort_values(
            'Call Queue Rank'
        )
        .head(20)
    )


## 15. Dynamic Voice AI Call Brief

Generate the structured context required by the voice agent: objective, recommended package, recommendation reason, and sales angle.


In [ ]:
# =========================
# 18.4) Dynamic call brief
# =========================


# -------------------------------------------------
# 1) التأكد من وجود الأعمدة المطلوبة
# -------------------------------------------------

required_call_brief_columns = [
    'Should Call',
    'Call Decision',
    'Call Decision Reason',

    'Call Objective',
    'Call Objective Details',

    'Package Recommendation Action',
    'Recommended Package',
    'Package Recommendation Reason',
    'Last Known Package',

    'Call Priority Level',
    'Call Queue Rank'
]


missing_call_brief_columns = [
    column
    for column in required_call_brief_columns
    if column not in calling_layer.columns
]


if missing_call_brief_columns:

    raise KeyError(
        f'Missing columns required for dynamic call brief: '
        f'{missing_call_brief_columns}'
    )


# -------------------------------------------------
# 2) تجهيز النصوص ومنع القيم المفقودة
# -------------------------------------------------

call_brief_text_columns = [
    'Call Decision',
    'Call Decision Reason',

    'Call Objective',
    'Call Objective Details',

    'Package Recommendation Action',
    'Recommended Package',
    'Package Recommendation Reason',
    'Last Known Package',

    'Call Priority Level'
]


for column in call_brief_text_columns:

    calling_layer[column] = (
        calling_layer[column]
        .fillna('')
        .astype(str)
        .str.strip()
    )


# -------------------------------------------------
# 3) تحديد زاوية البيع وطريقة بدء المكالمة
# -------------------------------------------------

def build_dynamic_call_guidance(row):

    should_call = bool(
        row['Should Call']
    )

    package_action = row[
        'Package Recommendation Action'
    ]

    call_objective = row[
        'Call Objective'
    ]

    recommended_package = row[
        'Recommended Package'
    ]

    previous_package = row[
        'Last Known Package'
    ]

    package_reason = row[
        'Package Recommendation Reason'
    ]

    call_priority_level = row[
        'Call Priority Level'
    ]

    call_queue_rank = row[
        'Call Queue Rank'
    ]


    # ---------------------------------------------
    # العميل غير مدرج للاتصال
    # ---------------------------------------------

    if not should_call:

        return pd.Series({
            'Ready for Voice AI':
                False,

            'Call Sales Angle':
                'لا توجد زاوية بيع حاليًا',

            'Call Opening Guidance':
                'لا يتم إجراء مكالمة لهذا العميل حاليًا.',

            'Dynamic Call Brief':
                (
                    f'قرار الاتصال: {row["Call Decision"]}. '
                    f'السبب: {row["Call Decision Reason"]}'
                )
        })


    # ---------------------------------------------
    # أول اشتراك
    # ---------------------------------------------

    if package_action == 'عرض اشتراك لأول مرة':

        sales_angle = (
            'ربط الباقة باستخدام العميل الفعلي '
            'وإظهار القيمة التي سيحصل عليها من الاشتراك'
        )

        opening_guidance = (
            'ابدأ بسؤال قصير عن احتياجه الحالي للغسيل، '
            'ثم اربط إجابته باستخدامه السابق، '
            f'وقدّم {recommended_package} كخيار مناسب. '
            'لا تبدأ بالسعر وحده ولا تقرأ سبب الترشيح حرفيًا.'
        )


    # ---------------------------------------------
    # تجديد الباقة السابقة
    # ---------------------------------------------

    elif package_action == 'تجديد الباقة السابقة':

        sales_angle = (
            'تسهيل التجديد وإعادة تفعيل المزايا '
            'التي سبق للعميل الاستفادة منها'
        )

        opening_guidance = (
            'ابدأ بتذكير العميل بشكل طبيعي بأنه سبق أن استخدم '
            f'{recommended_package}، ثم اسأله عن تجربته '
            'وانتقل إلى إغلاق التجديد بدون ضغط.'
        )


    # ---------------------------------------------
    # تجديد مع ترقية
    # ---------------------------------------------

    elif (
        package_action
        ==
        'تجديد مع ترقية إلى باقة أعلى'
    ):

        sales_angle = (
            'تجديد الاشتراك مع توضيح أن الباقة الأعلى '
            'تتناسب أكثر مع الاستخدام الحالي'
        )

        opening_guidance = (
            f'ابدأ بعرض تجديد الاشتراك، ثم وضح أن استخدام العميل '
            f'الحالي يناسب {recommended_package} أكثر من '
            f'{previous_package}. '
            'اربط الترقية بالاحتياج الفعلي وليس بارتفاع السعر.'
        )


    # ---------------------------------------------
    # استرجاع العميل
    # ---------------------------------------------

    elif (
        package_action
        ==
        'استرجاع العميل وتجديد الباقة السابقة'
    ):

        sales_angle = (
            'استرجاع العلاقة مع العميل وإقناعه '
            'بالعودة إلى باقته السابقة'
        )

        opening_guidance = (
            'ابدأ بالترحيب وسؤال العميل عن سبب توقفه أو تجربته السابقة، '
            f'ثم اعرض عليه العودة إلى {recommended_package}. '
            'لا تبدأ بالضغط على التجديد مباشرة.'
        )


    # ---------------------------------------------
    # تجديد حسب الاستخدام الحالي
    # ---------------------------------------------

    elif (
        package_action
        ==
        'تجديد وتحديد الباقة حسب الاستخدام الحالي'
    ):

        sales_angle = (
            'فهم الاحتياج الحالي وتأكيد أن الباقة المقترحة '
            'هي الأنسب قبل إغلاق التجديد'
        )

        opening_guidance = (
            'ابدأ بسؤال تأهيلي مختصر عن احتياجه الحالي، '
            f'ثم اقترح {recommended_package} بناءً على إجابته. '
            'نوع الباقة السابقة غير مؤكد، لذلك لا تدّعي معرفته.'
        )


    # ---------------------------------------------
    # حالة احتياطية
    # ---------------------------------------------

    else:

        sales_angle = (
            'متابعة العميل حسب هدف المكالمة المحدد'
        )

        opening_guidance = (
            f'ابدأ بفهم احتياج العميل، ثم نفّذ هدف المكالمة: '
            f'{call_objective}.'
        )


    # ---------------------------------------------
    # الملخص الديناميكي المرسل لمنصة الاتصال
    # ---------------------------------------------

    previous_package_text = (
        previous_package
        if previous_package
        else
        'غير محددة'
    )

    call_queue_rank_text = (
        str(int(call_queue_rank))
        if pd.notna(call_queue_rank)
        else
        'غير محدد'
    )


    dynamic_call_brief = (
        f'هدف المكالمة: {call_objective}\n'
        f'الإجراء المطلوب: {package_action}\n'
        f'الباقة السابقة: {previous_package_text}\n'
        f'الباقة المقترحة: {recommended_package}\n'
        f'سبب اختيار الباقة: {package_reason}\n'
        f'زاوية البيع: {sales_angle}\n'
        f'مستوى الأولوية: {call_priority_level}\n'
        f'ترتيب العميل في قائمة الاتصال: {call_queue_rank_text}\n'
        f'تعليمات البدء: {opening_guidance}'
    )


    return pd.Series({
        'Ready for Voice AI':
            True,

        'Call Sales Angle':
            sales_angle,

        'Call Opening Guidance':
            opening_guidance,

        'Dynamic Call Brief':
            dynamic_call_brief
    })


# -------------------------------------------------
# 4) حذف النتائج القديمة عند إعادة التشغيل
# -------------------------------------------------

dynamic_call_brief_columns = [
    'Ready for Voice AI',
    'Call Sales Angle',
    'Call Opening Guidance',
    'Dynamic Call Brief'
]


existing_dynamic_call_brief_columns = [
    column
    for column in dynamic_call_brief_columns
    if column in calling_layer.columns
]


if existing_dynamic_call_brief_columns:

    calling_layer = calling_layer.drop(
        columns=existing_dynamic_call_brief_columns
    )


# -------------------------------------------------
# 5) تطبيق الملخص الديناميكي
# -------------------------------------------------

dynamic_call_guidance_result = (
    calling_layer.apply(
        build_dynamic_call_guidance,
        axis=1
    )
)


calling_layer = pd.concat(
    [
        calling_layer,
        dynamic_call_guidance_result
    ],
    axis=1
)


# -------------------------------------------------
# 6) فحوصات النتيجة
# -------------------------------------------------

print("Voice AI readiness:")

display(
    calling_layer[
        'Ready for Voice AI'
    ].value_counts()
)


print("\nCall sales angles for callable customers:")

display(
    calling_layer.loc[
        calling_layer[
            'Should Call'
        ],
        'Call Sales Angle'
    ].value_counts()
)


print("\nSample dynamic call briefs:")

with pd.option_context(
    'display.max_colwidth',
    None
):

    display(
        calling_layer.loc[
            calling_layer[
                'Should Call'
            ],
            [
                'customer_id',
                'Call Queue Rank',
                'Call Objective',
                'Recommended Package',
                'Call Sales Angle',
                'Call Opening Guidance',
                'Dynamic Call Brief'
            ]
        ]
        .sort_values(
            'Call Queue Rank'
        )
        .head(10)
    )


---

## 🔐 Portfolio Safety

This notebook is a sanitized version of a production-oriented workflow.

The public version intentionally excludes:
- customer-level outputs
- phone numbers and customer IDs
- production datasets
- private company identifiers
- operational report exports

The core feature engineering, model training, scoring, targeting, and lead-prioritization approach is preserved for portfolio review.
